# SAFIR: Original Ordered Fusion + Drug-Swap Augmentation

This notebook keeps the original global fusion `[hA, hB, hA*hB, |hA-hB|, hC]`, uses training-only drug-swap augmentation with probability 0.5, the simplified reconstruction/delta/residual-consistency loss, strict no-leakage CRISPR preprocessing, and uncertainty-only active acquisition.

**Drug-swap augmentation is applied only to the training loader.** Validation, fixed-K evaluation, and active simulation keep the original A/B orientation.

In [1]:
# ============================================================
# SAFIR Active Dose-Matrix Completion
# Full No-GNN model + fixed 70/10/20 block split + uncertainty-only acquisition.
# CRISPR Top1000/median/scaler are fitted on TRAIN CELLS ONLY.
#
# 本版本用于验证精简损失函数：
# - Full model: No-GNN + residual anchor + CRISPR + Random Mask-K sparse training
# - 固定无泄露 70/10/20 block split，不做消融和冷启动
# - 损失仅保留三类具有明确任务含义的约束：
#     L = L_reconstruction + lambda_delta * L_delta + lambda_r * L_residual_consistency
#   其中 L_reconstruction = L_missing + lambda_obs * L_observed
# - 删除原版 smoothness regularization 与 epsilon L2 regularization
# - 主动选点只运行 uncertainty strategy，不运行 random/high_response/residual_hotspot/surface_reconstruction
#
# 输出：
# - fixed-K 补全评估表
# - uncertainty-only 主动选点逐 block/逐 budget 指标、汇总表与轨迹
# ============================================================

import os
import re
import json
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


In [2]:
# ============================================================
# 1. Config
# ============================================================

FEATURE_SET = "crispr1000"

# 按你的本地路径设置
BASE_DIR = Path(r"D:/6.2")
PROJECT_DIR = BASE_DIR / "DecoDoseNet-main"

COMBO_FILE = PROJECT_DIR / "processed_combination_response_r095_clean.csv"
DRUG_FEATURE_FILE = PROJECT_DIR / "feature" / "Graphlet_features_6_standardized.csv"
NOLEAK_FEATURE_DIR = (
    PROJECT_DIR / "oneil_cell_features_for_decodosenet" / "no_leakage_blocksplit_seed42"
)
CELL_FEATURE_FILE = NOLEAK_FEATURE_DIR / "CRISPR_TopVar1000_TrainOnly_cell_features.csv"
SPLIT_REFERENCE_FILE = NOLEAK_FEATURE_DIR / "ONEIL_4x4_block_split_70_10_20_seed42.csv"
CRISPR_PREPROCESSOR_FILE = NOLEAK_FEATURE_DIR / "ONEIL_CRISPR_TopVar1000_train_only_preprocessor.joblib"

# 这些文件必须先由 01_ONEIL_CRISPR_Top1000_TrainOnly_NoLeakage_v2.ipynb 生成。

# w/o CRISPR cell feature 消融：使用 977 基因表达量替代 CRISPR。
# 按你的说明，ONEIL_CELL_LINE_EXPRESSION.csv 与 CRISPR_TopVar1000_TrainOnly_cell_features.csv 放在同一个目录。
EXPRESSION_CELL_FEATURE_FILE = (BASE_DIR / "oneil_cell_features_for_decodosenet" / "ready_for_03DualBranch" / "ONEIL_CELL_LINE_EXPRESSION.csv")
if not EXPRESSION_CELL_FEATURE_FILE.exists():
    # 兼容少数旧放置方式。
    _alt_expression_files = [
        PROJECT_DIR / "ONEIL_CELL_LINE_EXPRESSION.csv",
        Path("/mnt/data/ONEIL_CELL_LINE_EXPRESSION.csv"),
    ]
    for _p in _alt_expression_files:
        if _p.exists():
            EXPRESSION_CELL_FEATURE_FILE = _p
            break

RESULT_ROOT_DIR = PROJECT_DIR / "results" / "SAFIR_SwapAug_Unc_SimpleLoss_v2"
# Keep Windows output paths deliberately short to avoid legacy MAX_PATH issues.
RESULT_ROOT_DIR.mkdir(parents=True, exist_ok=True)

# 当前正在运行的 model_mode 会动态修改 RESULT_DIR / ACTIVE_CELL_FEATURE_FILE
RESULT_DIR = RESULT_ROOT_DIR
ACTIVE_CELL_FEATURE_FILE = CELL_FEATURE_FILE

RANDOM_STATE = 42
TEST_SIZE = 0.20
TEST_CELL_SIZE = 0.20
TEST_DRUG_PAIR_SIZE = 0.20      # 留药物组合：测试集药物对比例
TEST_DRUG_SIZE = 0.20           # 留单药物：测试集药物比例
TEST_DOSE_POINT_SIZE = 0.20      # 留剂量点：测试剂量位置比例，默认约 3/16 个位置

# 稀疏组合实验设置：
# K=0 表示不给任何组合剂量点，只靠 drug/cell/单药响应补全曲面。
# K=2/4/6/8/10/12/14/16 表示模拟只测 K 个组合剂量点，再补全完整 4x4 曲面。
# 本版额外加入高观测密度 K=10/12/14/16，用来验证“少量补全”到“近完整曲面校正”的性能变化。
# 训练列表中对较高 K 做重复采样，使模型更充分学习 observed-point-guided completion。
DEFAULT_TRAIN_OBSERVED_K_LIST = [0, 2, 4, 4, 6, 6, 8, 8, 10, 12]
TRAIN_OBSERVED_K_LIST = list(DEFAULT_TRAIN_OBSERVED_K_LIST)
EVAL_OBSERVED_K_LIST = [0, 2, 4, 6, 8, 10, 12, 14, 16]
EVAL_MASK_REPEATS = 3

# 4x4 剂量矩阵
GRID_SIZE_A = 4
GRID_SIZE_B = 4
N_GRID = GRID_SIZE_A * GRID_SIZE_B
REQUIRE_COMPLETE_4X4 = True

# 训练参数
NUM_EPOCHS = 120
BATCH_SIZE = 512
LR = 5e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 20
MIN_DELTA = 1e-6
GRAD_CLIP_NORM = 5.0
NUM_WORKERS = 0

# Drug-swap augmentation (training only).
# With probability 0.5, represent the same physical experiment as (B, A):
# drug features and single-agent axes are exchanged, while Y/masks are transposed.
# Validation, fixed-K evaluation, and active simulation are NOT augmented.
DRUG_SWAP_AUGMENTATION = True
DRUG_SWAP_PROB = 0.50

# 精简损失权重
# L = (L_missing + lambda_obs * L_observed)
#     + lambda_delta * L_delta
#     + lambda_residual * L_residual_consistency
LAMBDA_OBS_RECON = 0.50
LAMBDA_DELTA = 0.12
LAMBDA_RESIDUAL_CONSISTENCY = 0.10

# 模型维度
DRUG_EMB_DIM = 128
CELL_EMB_DIM = 128
GLOBAL_DIM = 256
OBS_EMB_DIM = 64
HIDDEN_DIM = 256
DROPOUT = 0.10

# Grid-GNN 增强参数：16 个剂量点作为节点，水平/垂直/对角邻接边做消息传播
GRID_GNN_LAYERS = 1
GRID_EDGE_EMB_DIM = 16
LOCAL_OBS_EMB_DIM = 32

# 输出设置
SAVE_MATCHED_POINTS = True
SAVE_BLOCK_TABLE = True
SAVE_PREDICTIONS = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
# ============================================================
# Full model only: No-GNN residual-aware matrix completion
# ============================================================
# 本代码只保留一个 full model：
# - No GridGNN
# - residual anchor: Pred = Base + epsilon
# - ORIGINAL ordered global fusion: [hA, hB, hA*hB, |hA-hB|, hC]
# - TRAINING-ONLY drug-swap augmentation (p=0.5)
# - CRISPR cell feature
# - random mask-K sparse-completion training
#
# 只跑 block_random_split 随机划分；不跑消融和冷启动。
# 训练 full-model ensemble，并仅使用 uncertainty-guided active acquisition。

MODEL_MODES = [
    "ordered_fusion_swapaug",
]

MODEL_CONFIGS = {
    "ordered_fusion_swapaug": {
        "desc": "Full No-GNN SAFIR with original ordered global fusion + drug-swap training augmentation + residual anchor + CRISPR + Random Mask-K; simplified reconstruction/delta/residual-consistency loss; uncertainty-only active acquisition under fixed no-leakage 70/10/20 block split",
        "use_grid_gnn": False,
        "use_local_obs": True,
        "use_residual_anchor": True,
        "cell_feature_type": "CRISPR_TopVar1000_TrainOnly",
        "train_observed_k_list": list(DEFAULT_TRAIN_OBSERVED_K_LIST),
        "active_strategies": ["uncertainty"],
        "main_experiment_type": "ordered_global_fusion_drug_swap_augmentation",
    },
}

RUN_SETTINGS = [
    "block_random_split",
]


In [4]:
# ============================================================
# SAFIR-Active settings
# ============================================================
# 目标：从“随机给 K 个点补全曲面”升级为“动态选择下一批最值得测的剂量点”。
# 这里使用 retrospective active-learning simulation：完整 4x4 曲面作为隐藏真值，
# 每轮由模型选择未测剂量点，然后从真值中揭示该点，更新 obs_mask/obs_values 后继续选择。

ACTIVE_ENABLE = True

# 用多个 seed 训练 ensemble，估计预测不确定性。为了先能跑通，默认 3 个；论文实验建议 5 个。
ACTIVE_ENSEMBLE_SIZE = 3
ACTIVE_ENSEMBLE_SEEDS = [42, 2024, 3407, 777, 1009]

# 初始实验设计：
# none      : 一开始不测点，由模型直接选第一批
# random2   : 随机先测 2 个点
# corners4  : 先测 4 个角点，低低/低高/高低/高高
# center4   : 先测中心 4 个点
ACTIVE_INITIAL_SEED_MODE = "random2"

# 每一轮主动选择几个新剂量点；4x4 矩阵较小，建议 1 或 2。
ACTIVE_BATCH_SIZE = 2

# 总实验预算，即最终已经真实测过的组合剂量点数量。
ACTIVE_TOTAL_BUDGETS = [2, 4, 6, 8, 10, 12]

# 每个 block 使用不同 random2 初始观测重复评估的次数。
ACTIVE_REPEATS = 3

# 主动选择策略：仅保留 uncertainty。
# 对每个未观测位置，使用 deep ensemble 预测标准差作为 acquisition score，
# 每轮选择标准差最大的 ACTIVE_BATCH_SIZE 个位置。
ACTIVE_STRATEGIES = ["uncertainty"]

# 该变量仅用于结果中的最佳剂量/高响应区域评价，不参与 uncertainty acquisition。
MAXIMIZE_RESPONSE = True

# 是否额外保存每一步选点轨迹。
SAVE_ACTIVE_TRAJECTORY = True

print("Using device:", DEVICE)
print("Combo file:", COMBO_FILE)
print("Drug feature file:", DRUG_FEATURE_FILE)
print("CRISPR cell feature file (TRAIN-only fitted):", CELL_FEATURE_FILE)
print("Fixed split reference:", SPLIT_REFERENCE_FILE)
print("Expression cell feature file:", EXPRESSION_CELL_FEATURE_FILE)
print("Result root dir:", RESULT_ROOT_DIR)
print("Model modes:", MODEL_MODES)
print("Run settings:", RUN_SETTINGS)


Using device: cuda
Combo file: D:\6.2\DecoDoseNet-main\processed_combination_response_r095_clean.csv
Drug feature file: D:\6.2\DecoDoseNet-main\feature\Graphlet_features_6_standardized.csv
CRISPR cell feature file (TRAIN-only fitted): D:\6.2\DecoDoseNet-main\oneil_cell_features_for_decodosenet\no_leakage_blocksplit_seed42\CRISPR_TopVar1000_TrainOnly_cell_features.csv
Fixed split reference: D:\6.2\DecoDoseNet-main\oneil_cell_features_for_decodosenet\no_leakage_blocksplit_seed42\ONEIL_4x4_block_split_70_10_20_seed42.csv
Expression cell feature file: D:\6.2\oneil_cell_features_for_decodosenet\ready_for_03DualBranch\ONEIL_CELL_LINE_EXPRESSION.csv
Result root dir: D:\6.2\DecoDoseNet-main\results\SAFIR_SwapAug_Unc_SimpleLoss_v2
Model modes: ['ordered_fusion_swapaug']
Run settings: ['block_random_split']


In [5]:
# ============================================================
# 2. Utilities
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass


def normalize_match_name(x):
    """
    药物名/细胞系名标准化匹配：
    - 大写
    - 去掉空格、横杠、下划线、点、斜杠、括号等符号

    例：
    MK-2206 <-> MK2206
    SN-38 <-> SN38
    Bortezomib <-> BORTEZOMIB
    """
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = re.sub(r"[\s\-_./()]+", "", x)
    return x


def make_block_key(drug_a, drug_b, cell):
    """与上游无泄露 CRISPR 预处理 Notebook 完全相同的 block key。"""
    return "||".join([
        normalize_match_name(drug_a),
        normalize_match_name(drug_b),
        normalize_match_name(cell),
    ])


def build_normalized_index_map(index_values, label="feature"):
    key_to_raw = {}
    duplicated = []

    for raw in index_values:
        key = normalize_match_name(raw)
        if key == "":
            continue
        if key in key_to_raw:
            duplicated.append((key, key_to_raw[key], raw))
        else:
            key_to_raw[key] = raw

    if len(duplicated) > 0:
        print(f"[Warning] {label} has duplicated normalized keys. Keep first occurrence.")
        print("Examples:", duplicated[:10])

    return key_to_raw


def find_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    return None


def safe_pcc(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return np.nan
    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def concordance_index(y_true, y_pred):
    """
    C-index：衡量预测排序和真实排序的一致性。
    这里每个 4x4 block 只有 16 个点，所以 O(n^2) 计算不会慢。
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    n = 0
    h_sum = 0.0

    for i in range(len(y_true)):
        for j in range(i + 1, len(y_true)):
            if y_true[i] == y_true[j]:
                continue
            n += 1
            true_order = y_true[i] - y_true[j]
            pred_order = y_pred[i] - y_pred[j]

            if pred_order == 0:
                h_sum += 0.5
            elif true_order * pred_order > 0:
                h_sum += 1.0

    if n == 0:
        return np.nan

    return h_sum / n


def adjacent_delta_corr(y_true_surface, y_pred_surface):
    yt = np.asarray(y_true_surface, dtype=float)
    yp = np.asarray(y_pred_surface, dtype=float)

    true_deltas = []
    pred_deltas = []

    true_deltas.extend((yt[1:, :] - yt[:-1, :]).ravel().tolist())
    true_deltas.extend((yt[:, 1:] - yt[:, :-1]).ravel().tolist())

    pred_deltas.extend((yp[1:, :] - yp[:-1, :]).ravel().tolist())
    pred_deltas.extend((yp[:, 1:] - yp[:, :-1]).ravel().tolist())

    return safe_pcc(true_deltas, pred_deltas)


def topk_hit_metrics(y_true_surface, y_pred_surface, k=3):
    yt = np.asarray(y_true_surface).reshape(-1)
    yp = np.asarray(y_pred_surface).reshape(-1)

    true_best = int(np.argmax(yt))
    pred_order = np.argsort(-yp)

    top1_hit = int(pred_order[0] == true_best)
    topk_hit = int(true_best in pred_order[:k])

    true_best_ij = np.unravel_index(true_best, (GRID_SIZE_A, GRID_SIZE_B))
    pred_best_ij = np.unravel_index(int(pred_order[0]), (GRID_SIZE_A, GRID_SIZE_B))
    best_dose_distance = float(
        abs(true_best_ij[0] - pred_best_ij[0]) + abs(true_best_ij[1] - pred_best_ij[1])
    )

    return {
        "top1_hit": top1_hit,
        f"top{k}_hit": topk_hit,
        "best_dose_distance": best_dose_distance,
        "true_best_flat": true_best,
        "pred_best_flat": int(pred_order[0]),
    }


def make_observed_mask(n_grid, k, rng):
    mask = np.zeros(n_grid, dtype=np.float32)
    if k <= 0:
        return mask
    k = min(k, n_grid)
    idx = rng.choice(n_grid, size=k, replace=False)
    mask[idx] = 1.0
    return mask


def surface_smoothness_np(surface):
    s = np.asarray(surface, dtype=float)
    vals = []
    vals.append(np.mean((s[1:, :] - s[:-1, :]) ** 2))
    vals.append(np.mean((s[:, 1:] - s[:, :-1]) ** 2))
    return float(np.mean(vals))


In [6]:
# ============================================================
# 3. Load raw data
# ============================================================

def load_drug_features(path):
    df = pd.read_csv(path)

    name_col = find_column(df, ["name", "drug", "drug_name", "Drug", "DrugName"])
    if name_col is None:
        # 如果没有 name 列，默认第一列是药物名
        name_col = df.columns[0]

    drop_cols = []
    for c in df.columns:
        if c == name_col:
            continue
        if c.lower() in ["smile", "smiles", "mol", "molecule"]:
            drop_cols.append(c)

    feature_cols = [c for c in df.columns if c not in [name_col] + drop_cols]
    feat_df = df[[name_col] + feature_cols].copy()
    feat_df = feat_df.set_index(name_col)

    # 只保留数值列
    feat_df = feat_df.apply(pd.to_numeric, errors="coerce")
    feat_df = feat_df.fillna(0.0).astype(np.float32)

    return feat_df


def load_cell_features(path):
    df = pd.read_csv(path)

    cell_col = find_column(df, ["cell_line", "Cell_Line", "cell", "Cell", "CELL_LINE"])
    if cell_col is None:
        cell_col = df.columns[0]

    feat_df = df.set_index(cell_col)
    feat_df = feat_df.apply(pd.to_numeric, errors="coerce")
    feat_df = feat_df.fillna(0.0).astype(np.float32)

    return feat_df


def load_raw_data():
    combo_df = pd.read_csv(COMBO_FILE)
    drug_features = load_drug_features(DRUG_FEATURE_FILE)
    cell_features = load_cell_features(ACTIVE_CELL_FEATURE_FILE)

    print("\n========== Raw data ==========")
    print("Combo shape:", combo_df.shape)
    print("Combo columns:", combo_df.columns.tolist())
    print("Drug numeric feature shape:", drug_features.shape)
    print("Cell numeric feature shape:", cell_features.shape)

    return combo_df, drug_features, cell_features


In [7]:
# ============================================================
# 4. Match point-level combination data
# ============================================================

def preprocess_combo_points(combo_df, drug_features, cell_features):
    drug1_col = find_column(combo_df, ["drugA_name", "drug1_name", "drug_row", "Drug1", "drug1"])
    drug2_col = find_column(combo_df, ["drugB_name", "drug2_name", "drug_col", "Drug2", "drug2"])
    cell_col = find_column(combo_df, ["cell_line", "cell", "CellLine", "CELL_LINE"])

    conc1_col = find_column(combo_df, ["drugA_conc", "drug1_conc", "conc1", "dose1", "Drug1.Conc"])
    conc2_col = find_column(combo_df, ["drugB_conc", "drug2_conc", "conc2", "dose2", "Drug2.Conc"])

    single1_col = find_column(combo_df, ["single_resp_1", "single_response_1", "singleA", "YA"])
    single2_col = find_column(combo_df, ["single_resp_2", "single_response_2", "singleB", "YB"])

    target_col = find_column(combo_df, ["target", "response", "Y", "X/X0", "inhibition"])

    required = [drug1_col, drug2_col, cell_col, conc1_col, conc2_col, single1_col, single2_col, target_col]
    if any(c is None for c in required):
        raise ValueError(
            "找不到必要列，请检查组合数据列名。当前列：\n"
            + str(combo_df.columns.tolist())
        )

    drug_key_to_raw = build_normalized_index_map(drug_features.index, label="drug_features")
    cell_key_to_raw = build_normalized_index_map(cell_features.index, label="cell_features")

    combo_drugs = sorted(
        set(combo_df[drug1_col].dropna().astype(str)) |
        set(combo_df[drug2_col].dropna().astype(str))
    )
    exact_drug_match = len(set(combo_drugs) & set(map(str, drug_features.index)))
    norm_drug_match = sum(1 for d in combo_drugs if normalize_match_name(d) in drug_key_to_raw)

    print("\n========== Drug name matching check ==========")
    print("Combo unique drugs:", len(combo_drugs))
    print("Drug feature rows:", len(drug_features))
    print("Exact matched combo drugs:", exact_drug_match)
    print("Normalized matched combo drugs:", norm_drug_match)

    norm_missing_drugs = [d for d in combo_drugs if normalize_match_name(d) not in drug_key_to_raw]
    if len(norm_missing_drugs) > 0:
        print("[Warning] Still missing drugs after normalized matching:")
        print(norm_missing_drugs)
    else:
        print("All combo drugs matched after normalized matching.")

    rows = []
    skipped = {"drugA": 0, "drugB": 0, "cell": 0, "error": 0}

    for _, row in combo_df.iterrows():
        try:
            drugA = str(row[drug1_col]).strip()
            drugB = str(row[drug2_col]).strip()
            cell = str(row[cell_col]).strip()

            drugA_key = normalize_match_name(drugA)
            drugB_key = normalize_match_name(drugB)
            cell_key = normalize_match_name(cell)

            if drugA_key not in drug_key_to_raw:
                skipped["drugA"] += 1
                continue
            if drugB_key not in drug_key_to_raw:
                skipped["drugB"] += 1
                continue
            if cell_key not in cell_key_to_raw:
                skipped["cell"] += 1
                continue

            drugA_feature_name = drug_key_to_raw[drugA_key]
            drugB_feature_name = drug_key_to_raw[drugB_key]
            cell_feature_name = cell_key_to_raw[cell_key]

            drugA_conc = float(row[conc1_col])
            drugB_conc = float(row[conc2_col])
            single_resp_1 = float(row[single1_col])
            single_resp_2 = float(row[single2_col])
            target = float(row[target_col])

            if not np.isfinite([drugA_conc, drugB_conc, single_resp_1, single_resp_2, target]).all():
                skipped["error"] += 1
                continue

            rows.append({
                "drugA_name": drugA,
                "drugB_name": drugB,
                "cell_line": cell,
                "drugA_feature_name": drugA_feature_name,
                "drugB_feature_name": drugB_feature_name,
                "cell_feature_name": cell_feature_name,
                "drugA_conc": drugA_conc,
                "drugB_conc": drugB_conc,
                "single_resp_1": single_resp_1,
                "single_resp_2": single_resp_2,
                "target": target,
            })
        except Exception:
            skipped["error"] += 1

    data_df = pd.DataFrame(rows)

    print("\n========== Matched point-level data ==========")
    print("Valid point samples:", len(data_df))
    print("Valid cell lines:", data_df["cell_line"].nunique() if len(data_df) else 0)
    print("Valid drugA unique:", data_df["drugA_name"].nunique() if len(data_df) else 0)
    print("Valid drugB unique:", data_df["drugB_name"].nunique() if len(data_df) else 0)
    print("Skipped:", skipped)

    if SAVE_MATCHED_POINTS:
        out_path = RESULT_DIR / "matched_point_level_data.csv"
        data_df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print("Saved matched point-level data:", out_path)

    return data_df


In [8]:
# ============================================================
# 5. Build block-level 4x4 surface data
# ============================================================

def build_grid_features_from_axes(doseA_levels, doseB_levels, ya, yb):
    """
    Build the exact 17-D point-wise grid features used by SAFIR.

    Keeping this logic in one helper is important for drug-swap augmentation:
    after swapping A/B we REBUILD the grid features from the swapped dose and
    single-agent axes rather than trying to permute/sign-flip individual columns.
    """
    doseA_levels = np.asarray(doseA_levels, dtype=np.float32)
    doseB_levels = np.asarray(doseB_levels, dtype=np.float32)
    ya = np.asarray(ya, dtype=np.float32)
    yb = np.asarray(yb, dtype=np.float32)

    if len(doseA_levels) != GRID_SIZE_A or len(doseB_levels) != GRID_SIZE_B:
        raise ValueError(
            f"Expected {GRID_SIZE_A}x{GRID_SIZE_B} dose axes, got "
            f"{len(doseA_levels)}x{len(doseB_levels)}"
        )

    logA = np.log10(doseA_levels + 1e-12)
    logB = np.log10(doseB_levels + 1e-12)
    logA_norm = (logA - logA.mean()) / (logA.std() + 1e-6)
    logB_norm = (logB - logB.mean()) / (logB.std() + 1e-6)

    grid_feats = []
    for i in range(GRID_SIZE_A):
        for j in range(GRID_SIZE_B):
            a = logA_norm[i]
            b = logB_norm[j]
            yai = ya[i]
            ybj = yb[j]

            bliss_like = yai + ybj - yai * ybj
            max_single = max(yai, ybj)
            min_single = min(yai, ybj)

            grid_feats.append([
                a,
                b,
                a * b,
                abs(a - b),
                a + b,
                yai,
                ybj,
                yai * ybj,
                abs(yai - ybj),
                yai + ybj,
                bliss_like,
                max_single,
                min_single,
                yai - ybj,
                (yai - ybj) ** 2,
                a ** 2,
                b ** 2,
            ])

    return np.asarray(grid_feats, dtype=np.float32)


def transpose_flat_grid(x):
    """Transpose a flattened GRID_SIZE_A x GRID_SIZE_B array.

    This code is for the O'Neil 4x4 experiment, so the transposed shape is the
    same.  An explicit guard prevents silently using it for a non-square grid.
    """
    if GRID_SIZE_A != GRID_SIZE_B:
        raise ValueError(
            "Drug-swap augmentation in this O'Neil script assumes a square grid. "
            f"Current grid is {GRID_SIZE_A}x{GRID_SIZE_B}."
        )
    arr = np.asarray(x)
    return arr.reshape(GRID_SIZE_A, GRID_SIZE_B).T.reshape(-1).copy()


def build_surface_blocks(point_df, drug_features, cell_features):
    """
    一个 block = 一个 drugA-drugB-cell_line 的 4x4 剂量响应矩阵。
    输出 list[dict]，每个 dict 包含：
    - drug/cell 特征
    - doseA/doseB
    - YA[4], YB[4]
    - Y[4,4]
    - grid_features[16, 10]
    """
    blocks = []
    skipped = {
        "not_4x4": 0,
        "missing_points": 0,
        "feature_error": 0,
    }

    group_cols = ["drugA_name", "drugB_name", "cell_line"]
    grouped = point_df.groupby(group_cols, sort=False)

    for (drugA, drugB, cell), g in grouped:
        try:
            doseA_levels = sorted(g["drugA_conc"].unique())
            doseB_levels = sorted(g["drugB_conc"].unique())

            if REQUIRE_COMPLETE_4X4:
                if len(doseA_levels) != GRID_SIZE_A or len(doseB_levels) != GRID_SIZE_B:
                    skipped["not_4x4"] += 1
                    continue

            # pivot target
            y_mat = g.pivot_table(
                index="drugA_conc",
                columns="drugB_conc",
                values="target",
                aggfunc="mean"
            ).reindex(index=doseA_levels, columns=doseB_levels)

            if y_mat.isna().any().any():
                skipped["missing_points"] += 1
                continue

            # 单药响应向量
            ya = (
                g.groupby("drugA_conc")["single_resp_1"]
                .mean()
                .reindex(doseA_levels)
                .values
                .astype(np.float32)
            )
            yb = (
                g.groupby("drugB_conc")["single_resp_2"]
                .mean()
                .reindex(doseB_levels)
                .values
                .astype(np.float32)
            )

            if np.any(~np.isfinite(ya)) or np.any(~np.isfinite(yb)):
                skipped["missing_points"] += 1
                continue

            # feature names should be constant inside block
            drugA_feature_name = g["drugA_feature_name"].iloc[0]
            drugB_feature_name = g["drugB_feature_name"].iloc[0]
            cell_feature_name = g["cell_feature_name"].iloc[0]

            drugA_feat = drug_features.loc[drugA_feature_name].values.astype(np.float32)
            drugB_feat = drug_features.loc[drugB_feature_name].values.astype(np.float32)
            cell_feat = cell_features.loc[cell_feature_name].values.astype(np.float32)

            y = y_mat.values.astype(np.float32)

            # Build the canonical A/B-oriented point-wise dose/single-agent features.
            grid_feats = build_grid_features_from_axes(
                doseA_levels=doseA_levels,
                doseB_levels=doseB_levels,
                ya=ya,
                yb=yb,
            )

            blocks.append({
                "block_id": len(blocks),
                "block_key": make_block_key(drugA, drugB, cell),
                "drugA_name": drugA,
                "drugB_name": drugB,
                "cell_line": cell,
                "drugA_feature_name": drugA_feature_name,
                "drugB_feature_name": drugB_feature_name,
                "cell_feature_name": cell_feature_name,
                "doseA_levels": np.asarray(doseA_levels, dtype=np.float32),
                "doseB_levels": np.asarray(doseB_levels, dtype=np.float32),
                "YA": ya,
                "YB": yb,
                "Y": y,
                "grid_features": grid_feats,
                "drugA_feat": drugA_feat,
                "drugB_feat": drugB_feat,
                "cell_feat": cell_feat,
            })

        except Exception:
            skipped["feature_error"] += 1

    print("\n========== Block-level surface data ==========")
    print("Valid 4x4 blocks:", len(blocks))
    print("Skipped:", skipped)

    if len(blocks) == 0:
        raise RuntimeError("没有构建出任何 4x4 block，请检查数据。")

    block_summary = []
    for b in blocks:
        block_summary.append({
            "block_id": b["block_id"],
            "block_key": b["block_key"],
            "drugA_name": b["drugA_name"],
            "drugB_name": b["drugB_name"],
            "cell_line": b["cell_line"],
            "target_min": float(np.min(b["Y"])),
            "target_max": float(np.max(b["Y"])),
            "target_range": float(np.max(b["Y"]) - np.min(b["Y"])),
            "doseA_levels": "|".join([f"{x:g}" for x in b["doseA_levels"]]),
            "doseB_levels": "|".join([f"{x:g}" for x in b["doseB_levels"]]),
        })

    block_summary_df = pd.DataFrame(block_summary)

    print("Cell lines:", block_summary_df["cell_line"].nunique())
    print("Drug A unique:", block_summary_df["drugA_name"].nunique())
    print("Drug B unique:", block_summary_df["drugB_name"].nunique())
    print("Median target range:", block_summary_df["target_range"].median())

    if SAVE_BLOCK_TABLE:
        out_path = RESULT_DIR / "surface_block_summary.csv"
        block_summary_df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print("Saved block summary:", out_path)

    return blocks, block_summary_df


In [9]:
# ============================================================
# 6. Dataset
# ============================================================

class SurfaceBlockDataset(Dataset):
    def __init__(
        self,
        blocks,
        observed_k_list,
        training=True,
        random_state=42,
        drug_swap_augmentation=False,
        drug_swap_prob=0.50,
    ):
        self.blocks = blocks
        self.observed_k_list = list(observed_k_list)
        self.training = bool(training)
        self.drug_swap_augmentation = bool(drug_swap_augmentation)
        self.drug_swap_prob = float(drug_swap_prob)

        if not (0.0 <= self.drug_swap_prob <= 1.0):
            raise ValueError(f"drug_swap_prob must be in [0,1], got {self.drug_swap_prob}")
        if self.drug_swap_augmentation and GRID_SIZE_A != GRID_SIZE_B:
            raise ValueError(
                "This O'Neil drug-swap augmentation implementation requires a square grid; "
                f"got {GRID_SIZE_A}x{GRID_SIZE_B}."
            )

        # Preserve the original mask-K random stream for fair comparison with the
        # no-augmentation model.  Swap decisions use a separate RNG.
        self.rng = np.random.default_rng(random_state)
        self.swap_rng = np.random.default_rng(int(random_state) + 104729)

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, idx):
        b = self.blocks[idx]

        # Random Mask-K is generated exactly as in the original model.
        if self.training:
            k = int(self.rng.choice(self.observed_k_list))
            obs_mask = make_observed_mask(N_GRID, k, self.rng)
        else:
            # Evaluation functions construct their own masks; dataset default is K=0.
            k = 0
            obs_mask = np.zeros(N_GRID, dtype=np.float32)

        # Drug-swap augmentation is TRAINING-ONLY and uses an independent RNG.
        # The same physical experiment is re-expressed as (B,A):
        #   drugA <-> drugB
        #   doseA <-> doseB
        #   YA <-> YB
        #   Y -> Y^T
        #   observation/loss masks -> transpose
        do_swap = (
            self.training
            and self.drug_swap_augmentation
            and (self.swap_rng.random() < self.drug_swap_prob)
        )

        if do_swap:
            drugA_feat = b["drugB_feat"].astype(np.float32)
            drugB_feat = b["drugA_feat"].astype(np.float32)
            YA = b["YB"].astype(np.float32)
            YB = b["YA"].astype(np.float32)
            Y = b["Y"].T.copy().astype(np.float32)

            # Rebuild orientation-dependent grid features after swapping axes.
            grid_features = build_grid_features_from_axes(
                doseA_levels=b["doseB_levels"],
                doseB_levels=b["doseA_levels"],
                ya=YA,
                yb=YB,
            )

            # Preserve the same physical context points after A/B exchange.
            obs_mask = transpose_flat_grid(obs_mask).astype(np.float32)
            loss_mask = transpose_flat_grid(
                b.get("loss_mask", np.ones(N_GRID, dtype=np.float32))
            ).astype(np.float32)
            eval_dose_mask = transpose_flat_grid(
                b.get("eval_dose_mask", np.zeros(N_GRID, dtype=np.float32))
            ).astype(np.float32)
        else:
            drugA_feat = b["drugA_feat"].astype(np.float32)
            drugB_feat = b["drugB_feat"].astype(np.float32)
            YA = b["YA"].astype(np.float32)
            YB = b["YB"].astype(np.float32)
            Y = b["Y"].astype(np.float32)
            grid_features = b["grid_features"].astype(np.float32)
            loss_mask = b.get(
                "loss_mask", np.ones(N_GRID, dtype=np.float32)
            ).astype(np.float32)
            eval_dose_mask = b.get(
                "eval_dose_mask", np.zeros(N_GRID, dtype=np.float32)
            ).astype(np.float32)

        y_flat = Y.reshape(-1).astype(np.float32)
        obs_values = y_flat * obs_mask

        item = {
            "block_id": np.int64(b["block_id"]),
            "drugA_feat": drugA_feat,
            "drugB_feat": drugB_feat,
            "cell_feat": b["cell_feat"].astype(np.float32),
            "grid_features": grid_features,
            "YA": YA,
            "YB": YB,
            "Y": Y,
            "Y_flat": y_flat,
            "obs_mask": obs_mask.astype(np.float32),
            "obs_values": obs_values.astype(np.float32),
            "loss_mask": loss_mask,
            "eval_dose_mask": eval_dose_mask,
            "observed_k": np.int64(k),
            "drug_swapped": np.int64(1 if do_swap else 0),
        }
        return item


def collate_fn(batch):
    out = {}
    keys = batch[0].keys()
    for k in keys:
        vals = [b[k] for b in batch]
        if k in ["block_id", "observed_k", "drug_swapped"]:
            out[k] = torch.tensor(vals, dtype=torch.long)
        else:
            out[k] = torch.tensor(np.stack(vals), dtype=torch.float32)
    return out


def make_loader(
    blocks, observed_k_list, training=True, shuffle=True,
    drug_swap_augmentation=False, drug_swap_prob=DRUG_SWAP_PROB,
):
    ds = SurfaceBlockDataset(
        blocks=blocks,
        observed_k_list=observed_k_list,
        training=training,
        random_state=RANDOM_STATE,
        drug_swap_augmentation=drug_swap_augmentation,
        drug_swap_prob=drug_swap_prob,
    )
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE.type == "cuda" else False,
        persistent_workers=True if NUM_WORKERS > 0 else False,
        collate_fn=collate_fn,
    )


In [10]:
# ============================================================
# 7. Model: SAFIR-Active Residual-aware GridGNN
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.0, use_bn=True):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class GridMessagePassingLayer(nn.Module):
    """
    Residual-aware masked Grid-GNN layer.

    核心思想：
    - 节点：4x4 曲面上的 16 个剂量点。
    - 边：使用 8 类有向剂量边，显式区分剂量升高/降低方向。
    - 消息：由 source node 发出，但根据 source 是否为 observed 点动态加权。
    - 作用：让 observed residual 信息沿剂量网格传播到 missing 点，而不是让所有节点无差别平均。
    """
    def __init__(self, hidden_dim, edge_emb_dim, dropout=0.0, num_edge_types=8):
        super().__init__()
        self.edge_emb = nn.Embedding(num_edge_types, edge_emb_dim)

        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim + edge_emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.gate_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_emb_dim + 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

        self.update_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h, edge_index, edge_type, obs_mask):
        """
        h: [B, 16, H]
        edge_index: [2, E]
        edge_type: [E]
        obs_mask: [B, 16], 1 表示该剂量点有真实组合响应观测，0 表示 missing
        """
        B, N, H = h.shape
        src = edge_index[0]
        dst = edge_index[1]
        E = src.numel()

        h_src = h[:, src, :]                                      # [B, E, H]
        h_dst = h[:, dst, :]                                      # [B, E, H]
        e = self.edge_emb(edge_type).unsqueeze(0).expand(B, E, -1) # [B, E, edge_dim]

        src_obs = obs_mask[:, src].unsqueeze(-1)                  # [B, E, 1]
        dst_obs = obs_mask[:, dst].unsqueeze(-1)                  # [B, E, 1]

        msg_input = torch.cat([h_src, e], dim=-1)
        msg = self.message_mlp(msg_input.reshape(B * E, -1)).reshape(B, E, H)

        gate_input = torch.cat([h_src, h_dst, e, src_obs, dst_obs], dim=-1)
        gate = self.gate_mlp(gate_input.reshape(B * E, -1)).reshape(B, E, 1)

        # observed source 是强信息源；missing source 只保留少量先验传播能力，防止 K=0 时图层完全失效。
        src_conf = 0.15 + 0.85 * src_obs
        edge_weight = gate * src_conf
        msg = msg * edge_weight

        agg = torch.zeros(B, N, H, device=h.device, dtype=h.dtype)
        agg.index_add_(1, dst, msg)

        deg = torch.zeros(B, N, 1, device=h.device, dtype=h.dtype)
        deg.index_add_(1, dst, edge_weight)
        agg = agg / deg.clamp(min=1.0)

        delta = self.update_mlp(torch.cat([h, agg], dim=-1).reshape(B * N, -1)).reshape(B, N, H)
        h = self.norm1(h + self.dropout(delta))

        f = self.ffn(h.reshape(B * N, H)).reshape(B, N, H)
        h = self.norm2(h + self.dropout(f))
        return h

class SAFIRDoseSurfaceCompletionNet(nn.Module):
    """
    SAFIR-Dose-GridGNN model_mode-aware model.

    支持四类核心消融：
    - use_grid_gnn=False：去掉 Grid-GNN 消息传播，16 个剂量点只经过独立 point head。
    - use_local_obs=False：去掉 local obs node injection，只保留全局 obs_encoder。
    - use_residual_anchor=False：去掉 pred = alpha*YA + beta*YB + epsilon，改为直接预测 target。
    - cell feature 替换由外部 ACTIVE_CELL_FEATURE_FILE 控制。
    """

    def __init__(self, drug_dim, cell_dim, grid_feat_dim,
                 use_grid_gnn=True,
                 use_local_obs=True,
                 use_residual_anchor=True):
        super().__init__()
        self.use_grid_gnn = bool(use_grid_gnn)
        self.use_local_obs = bool(use_local_obs)
        self.use_residual_anchor = bool(use_residual_anchor)

        self.drug_encoder = MLP(
            in_dim=drug_dim,
            hidden_dims=[512, 256],
            out_dim=DRUG_EMB_DIM,
            dropout=DROPOUT,
            use_bn=True,
        )

        self.cell_encoder = MLP(
            in_dim=cell_dim,
            hidden_dims=[512, 256],
            out_dim=CELL_EMB_DIM,
            dropout=DROPOUT,
            use_bn=True,
        )

        global_in_dim = DRUG_EMB_DIM * 4 + CELL_EMB_DIM
        self.global_encoder = MLP(
            in_dim=global_in_dim,
            hidden_dims=[384],
            out_dim=GLOBAL_DIM,
            dropout=DROPOUT,
            use_bn=True,
        )

        # 全局 observed branch：无论是否 local obs injection 都保留。
        self.obs_encoder = MLP(
            in_dim=N_GRID * 3 + 5,
            hidden_dims=[128],
            out_dim=OBS_EMB_DIM,
            dropout=DROPOUT,
            use_bn=True,
        )

        if self.use_local_obs:
            # 局部 observed branch：
            # 每个剂量点只知道自己的 observed mask、observed value、observed residual 及 residual magnitude。
            # 注意：未观测点的 obs_values 与 obs_residual 均为 0，不会泄漏 target。
            self.local_obs_encoder = nn.Sequential(
                nn.Linear(4, LOCAL_OBS_EMB_DIM),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(LOCAL_OBS_EMB_DIM, LOCAL_OBS_EMB_DIM),
                nn.ReLU(),
            )
            local_dim = LOCAL_OBS_EMB_DIM
        else:
            self.local_obs_encoder = None
            local_dim = 0

        if self.use_residual_anchor:
            # base branch：只用 drug/cell/global + 单药/剂量 grid feature，负责 alpha/beta 单药锚定。
            # 这一路不过 GridGNN，避免图传播破坏单药响应基础项。
            base_node_in_dim = GLOBAL_DIM + grid_feat_dim
            self.base_node_projector = nn.Sequential(
                nn.Linear(base_node_in_dim, HIDDEN_DIM),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
                nn.ReLU(),
            )

            # epsilon branch：真正使用 GridGNN。
            # 如果 use_grid_gnn=True，不把 h_obs_rep 直接广播给每个节点；
            # observed 信息只能通过 local_obs 和 masked GridGNN 在网格上传播。
            if self.use_grid_gnn:
                eps_node_in_dim = GLOBAL_DIM + grid_feat_dim + local_dim
            else:
                eps_node_in_dim = GLOBAL_DIM + OBS_EMB_DIM + grid_feat_dim + local_dim

            self.eps_node_projector = nn.Sequential(
                nn.Linear(eps_node_in_dim, HIDDEN_DIM),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
                nn.ReLU(),
            )

            self.node_projector = None
        else:
            # 非 residual anchor 消融：保留旧式单分支预测。
            node_in_dim = GLOBAL_DIM + OBS_EMB_DIM + grid_feat_dim + local_dim
            self.node_projector = nn.Sequential(
                nn.Linear(node_in_dim, HIDDEN_DIM),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
                nn.ReLU(),
            )
            self.base_node_projector = None
            self.eps_node_projector = None

        if self.use_grid_gnn:
            self.grid_gnn_layers = nn.ModuleList([
                GridMessagePassingLayer(
                    hidden_dim=HIDDEN_DIM,
                    edge_emb_dim=GRID_EDGE_EMB_DIM,
                    dropout=DROPOUT,
                    num_edge_types=8,
                )
                for _ in range(GRID_GNN_LAYERS)
            ])
        else:
            self.grid_gnn_layers = nn.ModuleList([])

        if self.use_residual_anchor:
            self.ab_head = nn.Linear(HIDDEN_DIM, 2)
            self.epsilon_head = nn.Linear(HIDDEN_DIM, 1)
            # observed-gating：根据全局 sparse observation 调节整体交互残差强度。
            # 这里只作为全局 gate，不直接广播 h_obs 给 GridGNN 节点。
            self.epsilon_gate = nn.Sequential(
                nn.Linear(OBS_EMB_DIM + GLOBAL_DIM, 128),
                nn.ReLU(),
                nn.Linear(128, 1),
                nn.Sigmoid(),
            )
            self.direct_head = None
        else:
            self.ab_head = None
            self.epsilon_head = None
            self.epsilon_gate = None
            self.direct_head = nn.Linear(HIDDEN_DIM, 1)
        edge_index, edge_type = self.build_grid_edges(GRID_SIZE_A, GRID_SIZE_B)
        self.register_buffer("edge_index", edge_index, persistent=False)
        self.register_buffer("edge_type", edge_type, persistent=False)

    @staticmethod
    def build_grid_edges(n_a, n_b):
        """
        构造 4x4 grid 的 8 类有向剂量边。
        node index = i * n_b + j

        edge type:
        0: drugB increase
        1: drugB decrease
        2: drugA increase
        3: drugA decrease
        4: drugA increase + drugB increase
        5: drugA decrease + drugB decrease
        6: drugA increase + drugB decrease
        7: drugA decrease + drugB increase
        """
        edges = []
        types = []

        def add_directed(u, v, t):
            edges.append((u, v))
            types.append(t)

        for i in range(n_a):
            for j in range(n_b):
                u = i * n_b + j

                # horizontal: drugB direction
                if j + 1 < n_b:
                    v = i * n_b + (j + 1)
                    add_directed(u, v, 0)  # B increase
                    add_directed(v, u, 1)  # B decrease

                # vertical: drugA direction
                if i + 1 < n_a:
                    v = (i + 1) * n_b + j
                    add_directed(u, v, 2)  # A increase
                    add_directed(v, u, 3)  # A decrease

                # diagonal down: both increase/decrease
                if i + 1 < n_a and j + 1 < n_b:
                    v = (i + 1) * n_b + (j + 1)
                    add_directed(u, v, 4)  # A increase, B increase
                    add_directed(v, u, 5)  # A decrease, B decrease

                # diagonal up: one increase, one decrease
                if i + 1 < n_a and j - 1 >= 0:
                    v = (i + 1) * n_b + (j - 1)
                    add_directed(u, v, 6)  # A increase, B decrease
                    add_directed(v, u, 7)  # A decrease, B increase

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_type = torch.tensor(types, dtype=torch.long)
        return edge_index, edge_type
    def encode_observed_points(self, grid_features, obs_values, obs_mask):
        """
        构造全局 observed embedding，并在 use_local_obs=True 时构造局部节点 observed embedding。
        注意：w/o local obs injection 仍然保留 obs_residual 进入全局 obs_encoder，
        这样消融只检验“节点级局部注入”的贡献，而不是完全去掉 sparse obs branch。
        """
        bliss_grid = grid_features[:, :, 10]
        obs_residual = (obs_values - bliss_grid) * obs_mask

        obs_count_raw = obs_mask.sum(dim=1, keepdim=True)
        obs_count = obs_count_raw.clamp(min=1.0)
        obs_mean = (obs_values * obs_mask).sum(dim=1, keepdim=True) / obs_count

        very_neg = torch.full_like(obs_values, -1e9)
        very_pos = torch.full_like(obs_values, 1e9)
        obs_max = torch.where(obs_mask > 0.5, obs_values, very_neg).max(dim=1, keepdim=True).values
        obs_min = torch.where(obs_mask > 0.5, obs_values, very_pos).min(dim=1, keepdim=True).values
        obs_max = torch.where(obs_count_raw > 0, obs_max, torch.zeros_like(obs_max))
        obs_min = torch.where(obs_count_raw > 0, obs_min, torch.zeros_like(obs_min))

        obs_var = (((obs_values - obs_mean) ** 2) * obs_mask).sum(dim=1, keepdim=True) / obs_count
        obs_std = torch.sqrt(obs_var + 1e-8)
        obs_ratio = obs_mask.mean(dim=1, keepdim=True)

        obs_summary = torch.cat([obs_mean, obs_max, obs_min, obs_std, obs_ratio], dim=1)
        obs_input = torch.cat([obs_values, obs_mask, obs_residual, obs_summary], dim=1)
        h_obs = self.obs_encoder(obs_input)

        if self.use_local_obs:
            local_obs_input = torch.stack([
                obs_mask,
                obs_values * obs_mask,
                obs_residual,
                torch.abs(obs_residual),
            ], dim=-1)  # [B, 16, 4]
            B, N, _ = local_obs_input.shape
            h_local_obs = self.local_obs_encoder(local_obs_input.reshape(B * N, -1)).reshape(B, N, -1)
        else:
            h_local_obs = None

        return h_obs, h_local_obs, obs_residual

    def forward(self, batch):
        drugA = batch["drugA_feat"]
        drugB = batch["drugB_feat"]
        cell = batch["cell_feat"]
        grid_features = batch["grid_features"]      # [B, 16, G]
        obs_values = batch["obs_values"]            # [B, 16], missing 点为 0
        obs_mask = batch["obs_mask"]                # [B, 16], observed=1, missing=0

        B = drugA.size(0)

        hA = self.drug_encoder(drugA)
        hB = self.drug_encoder(drugB)
        hC = self.cell_encoder(cell)

        # Keep the ORIGINAL information-rich ordered fusion.
        # Drug-order robustness is encouraged at the DATA level via training-only
        # drug-swap augmentation rather than by deleting hA/hB from this representation.
        drug_pair = torch.cat([
            hA,
            hB,
            hA * hB,
            torch.abs(hA - hB),
            hC,
        ], dim=1)

        h_global = self.global_encoder(drug_pair)  # [B, GLOBAL_DIM]
        h_obs, h_local_obs, obs_residual = self.encode_observed_points(grid_features, obs_values, obs_mask)

        h_global_rep = h_global.unsqueeze(1).expand(B, N_GRID, -1)
        h_obs_rep = h_obs.unsqueeze(1).expand(B, N_GRID, -1)

        if self.use_residual_anchor:
            # ------------------------------------------------------------
            # 1) base branch：不过 GridGNN，只学习 alpha/beta 单药锚定。
            # ------------------------------------------------------------
            base_node_input = torch.cat([h_global_rep, grid_features], dim=-1)
            h_base = self.base_node_projector(
                base_node_input.reshape(B * N_GRID, -1)
            ).reshape(B, N_GRID, HIDDEN_DIM)

            # ------------------------------------------------------------
            # 2) epsilon branch：GridGNN 只负责组合残差/epsilon 曲面。
            #    GridGNN 模式下，不把 h_obs_rep 直接复制给所有节点，
            #    防止模型绕过剂量网格传播。
            # ------------------------------------------------------------
            eps_node_inputs = [h_global_rep, grid_features]
            if not self.use_grid_gnn:
                eps_node_inputs.insert(1, h_obs_rep)
            if self.use_local_obs:
                eps_node_inputs.append(h_local_obs)

            eps_node_input = torch.cat(eps_node_inputs, dim=-1)
            h_eps = self.eps_node_projector(
                eps_node_input.reshape(B * N_GRID, -1)
            ).reshape(B, N_GRID, HIDDEN_DIM)

            if self.use_grid_gnn:
                for layer in self.grid_gnn_layers:
                    h_eps = layer(h_eps, self.edge_index, self.edge_type, obs_mask)

            base_hidden = h_base.reshape(B * N_GRID, HIDDEN_DIM)
            eps_hidden = h_eps.reshape(B * N_GRID, HIDDEN_DIM)

            ab_logits = self.ab_head(base_hidden).reshape(B, N_GRID, 2)
            ab = torch.softmax(ab_logits, dim=-1)
            alpha = ab[:, :, 0]
            beta = ab[:, :, 1]

            epsilon_base = self.epsilon_head(eps_hidden).reshape(B, N_GRID)
            gate_input = torch.cat([h_global, h_obs], dim=1)
            eps_gate = self.epsilon_gate(gate_input)  # [B, 1], 0-1
            epsilon = epsilon_base * (0.5 + 1.5 * eps_gate)

            YA_grid = grid_features[:, :, 5]
            YB_grid = grid_features[:, :, 6]
            base = alpha * YA_grid + beta * YB_grid
            pred = base + epsilon

        else:
            # w/o residual anchor：保留旧式 direct prediction，主要用于消融。
            node_inputs = [h_global_rep, h_obs_rep, grid_features]
            if self.use_local_obs:
                node_inputs.append(h_local_obs)
            node_input = torch.cat(node_inputs, dim=-1)
            h = self.node_projector(node_input.reshape(B * N_GRID, -1)).reshape(B, N_GRID, HIDDEN_DIM)

            if self.use_grid_gnn:
                for layer in self.grid_gnn_layers:
                    h = layer(h, self.edge_index, self.edge_type, obs_mask)

            point_hidden = h.reshape(B * N_GRID, HIDDEN_DIM)
            pred = self.direct_head(point_hidden).reshape(B, N_GRID)
            alpha = torch.zeros_like(pred)
            beta = torch.zeros_like(pred)
            base = torch.zeros_like(pred)
            epsilon = torch.zeros_like(pred)
            epsilon_base = torch.zeros_like(pred)

        return {
            "pred_flat": pred,
            "pred_surface": pred.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "epsilon_flat": epsilon,
            "epsilon_surface": epsilon.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "epsilon_base_surface": epsilon_base.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "alpha_surface": alpha.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "beta_surface": beta.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "base_surface": base.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "obs_residual_surface": obs_residual.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
        }


In [11]:
# ============================================================
# 8. Loss
# ============================================================

def delta_loss(pred_surface, true_surface, loss_mask=None):
    """
    相邻剂量变化趋势约束。
    若未来使用 dose holdout，只有相邻两个点都允许参与训练时才计算该差分，
    防止 held-out true Y 通过 delta loss 间接进入梯度。
    当前 block-random split 下 loss_mask 全1，因此不改变主实验定义。
    """
    dpa = pred_surface[:, 1:, :] - pred_surface[:, :-1, :]
    dta = true_surface[:, 1:, :] - true_surface[:, :-1, :]
    dpb = pred_surface[:, :, 1:] - pred_surface[:, :, :-1]
    dtb = true_surface[:, :, 1:] - true_surface[:, :, :-1]

    if loss_mask is None:
        return (F.mse_loss(dpa, dta) + F.mse_loss(dpb, dtb)) / 2.0

    mask = loss_mask.reshape_as(true_surface)
    ma = mask[:, 1:, :] * mask[:, :-1, :]
    mb = mask[:, :, 1:] * mask[:, :, :-1]

    la = (((dpa - dta) ** 2) * ma).sum() / ma.sum().clamp(min=1.0)
    lb = (((dpb - dtb) ** 2) * mb).sum() / mb.sum().clamp(min=1.0)
    return (la + lb) / 2.0



def compute_loss(outputs, batch):
    """
    Simplified SAFIR objective:
        L = L_reconstruction + lambda_delta * L_delta
            + lambda_residual * L_residual_consistency

    where
        L_reconstruction = L_missing + lambda_obs * L_observed.

    Removed from the original objective:
        - smoothness regularization
        - epsilon L2 regularization
    """
    pred = outputs["pred_flat"]
    y = batch["Y_flat"]
    obs_mask = batch["obs_mask"]
    loss_mask = batch.get("loss_mask", torch.ones_like(obs_mask))

    miss_mask = (1.0 - obs_mask) * loss_mask
    obs_loss_mask = obs_mask * loss_mask

    # 1) Reconstruction loss: missing positions are the primary target;
    #    observed positions receive a lower-weight consistency term.
    miss_count = miss_mask.sum().clamp(min=1.0)
    obs_count = obs_loss_mask.sum().clamp(min=1.0)

    loss_missing = (((pred - y) ** 2) * miss_mask).sum() / miss_count
    loss_obs = (((pred - y) ** 2) * obs_loss_mask).sum() / obs_count
    loss_reconstruction = loss_missing + LAMBDA_OBS_RECON * loss_obs

    # 2) Landscape local-difference consistency along the two dose axes.
    loss_delta = delta_loss(
        outputs["pred_surface"],
        batch["Y"],
        loss_mask=loss_mask,
    )

    # 3) Observation-guided residual consistency.
    #    epsilon should approximate Y - Base at observed positions.
    #    Base is detached so this auxiliary supervision only updates the residual branch.
    base_flat = outputs["base_surface"].reshape_as(y)
    if outputs["epsilon_flat"].requires_grad:
        true_obs_residual = (y - base_flat.detach()) * obs_loss_mask
        loss_residual_consistency = (
            ((outputs["epsilon_flat"] - true_obs_residual) ** 2) * obs_loss_mask
        ).sum() / obs_count
    else:
        loss_residual_consistency = pred.new_tensor(0.0)

    total = (
        loss_reconstruction
        + LAMBDA_DELTA * loss_delta
        + LAMBDA_RESIDUAL_CONSISTENCY * loss_residual_consistency
    )

    return total, {
        "loss_total": float(total.detach().cpu()),
        "loss_reconstruction": float(loss_reconstruction.detach().cpu()),
        "loss_missing": float(loss_missing.detach().cpu()),
        "loss_obs": float(loss_obs.detach().cpu()),
        "loss_delta": float(loss_delta.detach().cpu()),
        "loss_residual_consistency": float(loss_residual_consistency.detach().cpu()),
    }


In [12]:
# ============================================================
# 9. Split
# ============================================================

def load_fixed_no_leakage_block_split(blocks, split_reference_file=SPLIT_REFERENCE_FILE):
    """
    读取上游 CRISPR 预处理阶段已经固定好的 70/10/20 block split。
    绝不在这里重新随机划分；否则会破坏 train-only Top1000 的数据隔离。
    """
    if not Path(split_reference_file).exists():
        raise FileNotFoundError(
            f"固定 split 文件不存在：{split_reference_file}\n"
            "请先运行 01_ONEIL_CRISPR_Top1000_TrainOnly_NoLeakage_v2.ipynb"
        )

    split_df = pd.read_csv(split_reference_file)
    required = {"block_key", "split"}
    if not required.issubset(split_df.columns):
        raise ValueError(f"split 文件必须包含 {required}，当前列：{split_df.columns.tolist()}")

    split_df["block_key"] = split_df["block_key"].astype(str)
    split_df["split"] = split_df["split"].astype(str).str.lower().replace({"validation": "val"})
    if split_df["block_key"].duplicated().any():
        dup = split_df.loc[split_df["block_key"].duplicated(False), "block_key"].head(10).tolist()
        raise RuntimeError(f"split reference 中 block_key 重复：{dup}")

    split_map = dict(zip(split_df["block_key"], split_df["split"]))
    current_keys = {str(b["block_key"]) for b in blocks}
    ref_keys = set(split_map)

    missing_in_reference = sorted(current_keys - ref_keys)
    missing_in_current = sorted(ref_keys - current_keys)
    if missing_in_reference or missing_in_current:
        raise RuntimeError(
            "当前 SAFIR blocks 与生成 CRISPR 时的 blocks 不一致。\n"
            f"当前有但 split 无：{missing_in_reference[:10]}\n"
            f"split 有但当前无：{missing_in_current[:10]}\n"
            "请确保组合数据、Graphlet 文件和筛选条件与上游预处理 Notebook 相同。"
        )

    train_blocks = [b for b in blocks if split_map[b["block_key"]] == "train"]
    val_blocks = [b for b in blocks if split_map[b["block_key"]] == "val"]
    test_blocks = [b for b in blocks if split_map[b["block_key"]] == "test"]

    assigned = len(train_blocks) + len(val_blocks) + len(test_blocks)
    if assigned != len(blocks):
        unknown = sorted({split_map[b["block_key"]] for b in blocks} - {"train", "val", "test"})
        raise RuntimeError(f"存在未知 split 标签：{unknown}")
    if not train_blocks or not val_blocks or not test_blocks:
        raise RuntimeError("Train/Val/Test 任一集合为空。")

    # 保存本次实际使用的审计表
    audit_rows = []
    for b in blocks:
        audit_rows.append({
            "block_id_runtime": b["block_id"],
            "block_key": b["block_key"],
            "drugA_name": b["drugA_name"],
            "drugB_name": b["drugB_name"],
            "cell_line": b["cell_line"],
            "split": split_map[b["block_key"]],
        })
    pd.DataFrame(audit_rows).to_csv(
        RESULT_DIR / "block_split_reference_reused_from_CRISPR_preprocessing.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print("\n========== Fixed no-leakage block split ==========")
    print("Train blocks:", len(train_blocks))
    print("Val blocks:", len(val_blocks))
    print("Test blocks:", len(test_blocks))
    print("Split source:", split_reference_file)
    return train_blocks, val_blocks, test_blocks

def make_cell_line_cold_start_split(blocks):
    cells = sorted(set(b["cell_line"] for b in blocks))
    train_cells, test_cells = train_test_split(
        cells,
        test_size=TEST_CELL_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )
    train_cells = set(train_cells)
    test_cells = set(test_cells)

    train_blocks = [b for b in blocks if b["cell_line"] in train_cells]
    test_blocks = [b for b in blocks if b["cell_line"] in test_cells]

    print("\n========== Cell-line cold-start split ==========")
    print("Train cells:", len(train_cells), sorted(train_cells))
    print("Test cells:", len(test_cells), sorted(test_cells))
    print("Train blocks:", len(train_blocks))
    print("Test blocks:", len(test_blocks))

    split_df = pd.DataFrame({
        "cell_line": sorted(list(train_cells | test_cells)),
        "split": ["train" if c in train_cells else "test" for c in sorted(list(train_cells | test_cells))]
    })
    split_df.to_csv(RESULT_DIR / "cell_line_cold_start_split_reference.csv", index=False, encoding="utf-8-sig")

    return train_blocks, test_blocks


def canonical_drug_pair(drug_a, drug_b):
    """
    用于留药物组合冷启动：
    将 drugA-drugB 和 drugB-drugA 视为同一个组合，避免反向组合泄漏。
    """
    a = normalize_match_name(drug_a)
    b = normalize_match_name(drug_b)
    return tuple(sorted([a, b]))


def make_drug_combination_cold_start_split(blocks):
    """
    留药物组合 cold-start：
    - 测试集中的 drugA-drugB 组合在训练集中完全不出现；
    - 但单个药物本身可以在训练集中通过其他组合出现；
    - 用来评估模型对“未见过药物配对”的泛化能力。

    注意：这里按 unordered pair 划分，因此 A-B 与 B-A 会被放在同一 split，避免组合反向泄漏。
    """
    pair_to_blocks = {}
    pair_to_raw_names = {}

    for b in blocks:
        pair = canonical_drug_pair(b["drugA_name"], b["drugB_name"])
        pair_to_blocks.setdefault(pair, []).append(b)
        pair_to_raw_names[pair] = tuple(sorted([str(b["drugA_name"]), str(b["drugB_name"])]))

    pairs = sorted(pair_to_blocks.keys())

    train_pairs, test_pairs = train_test_split(
        pairs,
        test_size=TEST_DRUG_PAIR_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    train_pairs = set(train_pairs)
    test_pairs = set(test_pairs)

    train_blocks = []
    test_blocks = []
    for pair, bs in pair_to_blocks.items():
        if pair in train_pairs:
            train_blocks.extend(bs)
        elif pair in test_pairs:
            test_blocks.extend(bs)

    train_pair_check = set(canonical_drug_pair(b["drugA_name"], b["drugB_name"]) for b in train_blocks)
    test_pair_check = set(canonical_drug_pair(b["drugA_name"], b["drugB_name"]) for b in test_blocks)
    overlap = train_pair_check & test_pair_check

    print("\n========== Drug-combination cold-start split ==========")
    print("Train drug pairs:", len(train_pairs))
    print("Test drug pairs:", len(test_pairs))
    print("Train blocks:", len(train_blocks))
    print("Test blocks:", len(test_blocks))
    print("Pair overlap between train/test:", len(overlap))

    if len(overlap) > 0:
        raise RuntimeError(f"Drug-combination cold-start leakage detected: {len(overlap)} overlapping pairs")

    split_rows = []
    for pair in pairs:
        raw_a, raw_b = pair_to_raw_names[pair]
        split_rows.append({
            "drug_pair_key": " + ".join(pair),
            "drugA_name_sorted": raw_a,
            "drugB_name_sorted": raw_b,
            "n_blocks": len(pair_to_blocks[pair]),
            "split": "train" if pair in train_pairs else "test"
        })

    pd.DataFrame(split_rows).to_csv(
        RESULT_DIR / "drug_combination_cold_start_split_reference.csv",
        index=False,
        encoding="utf-8-sig"
    )

    return train_blocks, test_blocks


def make_single_drug_cold_start_split(blocks):
    """
    留单药物 cold-start：
    - 随机抽取一部分药物作为 held-out drugs；
    - 训练集只包含 drugA 和 drugB 都不属于 held-out drugs 的 block；
    - 测试集包含至少一个药物属于 held-out drugs 的 block；
    - 因此测试集中会出现训练阶段完全没见过的单个药物。

    这个设置比“留药物组合”更难，因为模型需要泛化到新药物本身。
    """
    all_drugs = sorted(set(
        [normalize_match_name(b["drugA_name"]) for b in blocks] +
        [normalize_match_name(b["drugB_name"]) for b in blocks]
    ))

    train_drugs, test_drugs = train_test_split(
        all_drugs,
        test_size=TEST_DRUG_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    train_drugs = set(train_drugs)
    test_drugs = set(test_drugs)

    train_blocks = []
    test_blocks = []

    for b in blocks:
        da = normalize_match_name(b["drugA_name"])
        db = normalize_match_name(b["drugB_name"])

        if da in test_drugs or db in test_drugs:
            test_blocks.append(b)
        else:
            train_blocks.append(b)

    train_drug_check = set(
        [normalize_match_name(b["drugA_name"]) for b in train_blocks] +
        [normalize_match_name(b["drugB_name"]) for b in train_blocks]
    )
    overlap = train_drug_check & test_drugs

    print("\n========== Single-drug cold-start split ==========")
    print("Train drugs:", len(train_drugs), sorted(train_drugs))
    print("Held-out test drugs:", len(test_drugs), sorted(test_drugs))
    print("Train blocks:", len(train_blocks))
    print("Test blocks:", len(test_blocks))
    print("Held-out drug leakage in train:", len(overlap))

    if len(overlap) > 0:
        raise RuntimeError(f"Single-drug cold-start leakage detected: {len(overlap)} held-out drugs in train")

    split_df = pd.DataFrame({
        "drug_key": all_drugs,
        "split": ["test" if d in test_drugs else "train" for d in all_drugs]
    })
    split_df.to_csv(
        RESULT_DIR / "single_drug_cold_start_split_reference.csv",
        index=False,
        encoding="utf-8-sig"
    )

    return train_blocks, test_blocks


def clone_blocks_with_masks(blocks, loss_mask=None, eval_dose_mask=None):
    """
    为 dose-point cold-start 创建带 loss/eval mask 的 block 副本，
    避免同一批 blocks 在不同 setting 之间互相污染。
    """
    out = []
    if loss_mask is None:
        loss_mask = np.ones(N_GRID, dtype=np.float32)
    if eval_dose_mask is None:
        eval_dose_mask = np.zeros(N_GRID, dtype=np.float32)
    loss_mask = np.asarray(loss_mask, dtype=np.float32)
    eval_dose_mask = np.asarray(eval_dose_mask, dtype=np.float32)
    for b in blocks:
        nb = dict(b)
        nb["loss_mask"] = loss_mask.copy()
        nb["eval_dose_mask"] = eval_dose_mask.copy()
        out.append(nb)
    return out


def make_dose_point_cold_start_split(blocks):
    """
    留剂量点 cold-start：
    - 按比例从 16 个剂量网格位置中抽取 held-out dose positions；
    - 训练阶段这些剂量位置不参与监督损失，模拟模型从未在训练监督中见过这些剂量坐标上的组合响应；
    - 测试阶段额外报告 heldout-dose 指标，用来衡量未见剂量位置的泛化。

    这里同时做 block 级 8:2 划分，避免测试 block 完全出现在训练中；
    因此该 setting 是“新 block + 未见剂量位置”的更严格泛化评估。
    """
    all_dose_idx = np.arange(N_GRID)
    train_dose_idx, test_dose_idx = train_test_split(
        all_dose_idx,
        test_size=TEST_DOSE_POINT_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )
    train_dose_idx = np.array(sorted(train_dose_idx.tolist()), dtype=int)
    test_dose_idx = np.array(sorted(test_dose_idx.tolist()), dtype=int)

    loss_mask = np.ones(N_GRID, dtype=np.float32)
    loss_mask[test_dose_idx] = 0.0
    eval_dose_mask = np.zeros(N_GRID, dtype=np.float32)
    eval_dose_mask[test_dose_idx] = 1.0

    idx = np.arange(len(blocks))
    train_idx, test_idx = train_test_split(
        idx,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )
    train_blocks_raw = [blocks[i] for i in train_idx]
    test_blocks_raw = [blocks[i] for i in test_idx]

    train_blocks = clone_blocks_with_masks(train_blocks_raw, loss_mask=loss_mask, eval_dose_mask=eval_dose_mask)
    test_blocks = clone_blocks_with_masks(test_blocks_raw, loss_mask=np.ones(N_GRID, dtype=np.float32), eval_dose_mask=eval_dose_mask)

    print("\n========== Dose-point cold-start split ==========")
    print("Held-out dose positions:", test_dose_idx.tolist())
    print("Train dose positions supervised:", train_dose_idx.tolist())
    print("Train blocks:", len(train_blocks))
    print("Test blocks:", len(test_blocks))

    split_rows = []
    test_dose_set = set(test_dose_idx.tolist())
    for flat_idx in all_dose_idx:
        i, j = flat_to_ij(int(flat_idx))
        split_rows.append({
            "flat_idx": int(flat_idx),
            "doseA_index": i,
            "doseB_index": j,
            "split": "test_heldout" if int(flat_idx) in test_dose_set else "train_supervised",
        })
    pd.DataFrame(split_rows).to_csv(
        RESULT_DIR / "dose_point_cold_start_split_reference.csv",
        index=False,
        encoding="utf-8-sig",
    )

    return train_blocks, test_blocks


In [13]:
# ============================================================
# 10. Evaluation
# ============================================================

@torch.no_grad()
def predict_blocks_with_mask(model, blocks, observed_k=0, mask_repeat=1, seed=42):
    model.eval()
    rng = np.random.default_rng(seed)

    records = []
    metric_rows = []

    for repeat in range(mask_repeat):
        for b in blocks:
            y_flat = b["Y"].reshape(-1).astype(np.float32)
            obs_mask = make_observed_mask(N_GRID, observed_k, rng)
            obs_values = y_flat * obs_mask

            batch = {
                "block_id": torch.tensor([b["block_id"]], dtype=torch.long, device=DEVICE),
                "drugA_feat": torch.tensor(b["drugA_feat"][None, :], dtype=torch.float32, device=DEVICE),
                "drugB_feat": torch.tensor(b["drugB_feat"][None, :], dtype=torch.float32, device=DEVICE),
                "cell_feat": torch.tensor(b["cell_feat"][None, :], dtype=torch.float32, device=DEVICE),
                "grid_features": torch.tensor(b["grid_features"][None, :, :], dtype=torch.float32, device=DEVICE),
                "YA": torch.tensor(b["YA"][None, :], dtype=torch.float32, device=DEVICE),
                "YB": torch.tensor(b["YB"][None, :], dtype=torch.float32, device=DEVICE),
                "Y": torch.tensor(b["Y"][None, :, :], dtype=torch.float32, device=DEVICE),
                "Y_flat": torch.tensor(y_flat[None, :], dtype=torch.float32, device=DEVICE),
                "obs_mask": torch.tensor(obs_mask[None, :], dtype=torch.float32, device=DEVICE),
                "obs_values": torch.tensor(obs_values[None, :], dtype=torch.float32, device=DEVICE),
                "loss_mask": torch.tensor(b.get("loss_mask", np.ones(N_GRID, dtype=np.float32))[None, :], dtype=torch.float32, device=DEVICE),
                "eval_dose_mask": torch.tensor(b.get("eval_dose_mask", np.zeros(N_GRID, dtype=np.float32))[None, :], dtype=torch.float32, device=DEVICE),
                "observed_k": torch.tensor([observed_k], dtype=torch.long, device=DEVICE),
            }

            out = model(batch)

            pred = out["pred_surface"][0].detach().cpu().numpy()
            eps = out["epsilon_surface"][0].detach().cpu().numpy()
            alpha = out["alpha_surface"][0].detach().cpu().numpy()
            beta = out["beta_surface"][0].detach().cpu().numpy()
            base = out["base_surface"][0].detach().cpu().numpy()
            y = b["Y"]

            miss_mask = 1.0 - obs_mask
            all_true = y.reshape(-1)
            all_pred = pred.reshape(-1)

            if miss_mask.sum() > 0:
                miss_true = all_true[miss_mask.astype(bool)]
                miss_pred = all_pred[miss_mask.astype(bool)]
                rmse_missing = float(np.sqrt(mean_squared_error(miss_true, miss_pred)))
                mae_missing = float(mean_absolute_error(miss_true, miss_pred))
                pcc_missing = safe_pcc(miss_true, miss_pred)
                c_index_missing = concordance_index(miss_true, miss_pred)
            else:
                rmse_missing = np.nan
                mae_missing = np.nan
                pcc_missing = np.nan
                c_index_missing = np.nan

            hit = topk_hit_metrics(y, pred, k=3)

            metric_rows.append({
                "repeat": repeat,
                "block_id": b["block_id"],
                "drugA_name": b["drugA_name"],
                "drugB_name": b["drugB_name"],
                "cell_line": b["cell_line"],
                "observed_k": observed_k,
                "rmse_all": float(np.sqrt(mean_squared_error(all_true, all_pred))),
                "mae_all": float(mean_absolute_error(all_true, all_pred)),
                "pcc_all": safe_pcc(all_true, all_pred),
                "r2_all": float(r2_score(all_true, all_pred)) if np.std(all_true) > 1e-12 else np.nan,
                "c_index_all": concordance_index(all_true, all_pred),
                "rmse_missing": rmse_missing,
                "mae_missing": mae_missing,
                "pcc_missing": pcc_missing,
                "c_index_missing": c_index_missing,
                "adjacent_delta_corr": adjacent_delta_corr(y, pred),
                "true_surface_smoothness": surface_smoothness_np(y),
                "pred_surface_smoothness": surface_smoothness_np(pred),
                "epsilon_surface_smoothness": surface_smoothness_np(eps),
                "top1_hit": hit["top1_hit"],
                "top3_hit": hit["top3_hit"],
                "best_dose_distance": hit["best_dose_distance"],
                "true_best_flat": hit["true_best_flat"],
                "pred_best_flat": hit["pred_best_flat"],
            })

            if SAVE_PREDICTIONS:
                for i in range(GRID_SIZE_A):
                    for j in range(GRID_SIZE_B):
                        flat_idx = i * GRID_SIZE_B + j
                        records.append({
                            "repeat": repeat,
                            "block_id": b["block_id"],
                            "drugA_name": b["drugA_name"],
                            "drugB_name": b["drugB_name"],
                            "cell_line": b["cell_line"],
                            "observed_k": observed_k,
                            "doseA_index": i,
                            "doseB_index": j,
                            "doseA": float(b["doseA_levels"][i]),
                            "doseB": float(b["doseB_levels"][j]),
                            "is_observed_input": int(obs_mask[flat_idx] > 0.5),
                            "single_resp_1": float(b["YA"][i]),
                            "single_resp_2": float(b["YB"][j]),
                            "target": float(y[i, j]),
                            "prediction": float(pred[i, j]),
                            "base": float(base[i, j]),
                            "epsilon": float(eps[i, j]),
                            "alpha": float(alpha[i, j]),
                            "beta": float(beta[i, j]),
                        })

    metrics_df = pd.DataFrame(metric_rows)
    pred_df = pd.DataFrame(records) if SAVE_PREDICTIONS else None

    return metrics_df, pred_df


def summarize_metrics(metrics_df):
    agg_spec = {
        "rmse_all": "mean",
        "mae_all": "mean",
        "pcc_all": "mean",
        "r2_all": "mean",
        "c_index_all": "mean",
        "rmse_missing": "mean",
        "mae_missing": "mean",
        "pcc_missing": "mean",
        "c_index_missing": "mean",
        "adjacent_delta_corr": "mean",
        "top1_hit": "mean",
        "top3_hit": "mean",
        "best_dose_distance": "mean",
        "rmse_heldout_dose": "mean",
        "mae_heldout_dose": "mean",
        "pcc_heldout_dose": "mean",
    }
    agg_spec = {k: v for k, v in agg_spec.items() if k in metrics_df.columns}
    agg = metrics_df.groupby("observed_k").agg(agg_spec).reset_index()
    return agg


@torch.no_grad()
def evaluate_val_loss(model, val_loader):
    model.eval()
    losses = []

    for batch in val_loader:
        batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
        out = model(batch)
        loss, _ = compute_loss(out, batch)
        losses.append(float(loss.detach().cpu()))

    return float(np.mean(losses))


In [14]:
# ============================================================
# 11. SAFIR-Active: training + dynamic dose-point selection
# ============================================================


def make_loader_with_seed(
    blocks, observed_k_list, training=True, shuffle=True, seed=42,
    drug_swap_augmentation=False, drug_swap_prob=DRUG_SWAP_PROB,
):
    """Loader with independent mask/swap RNG streams for each ensemble seed."""
    ds = SurfaceBlockDataset(
        blocks=blocks,
        observed_k_list=observed_k_list,
        training=training,
        random_state=seed,
        drug_swap_augmentation=drug_swap_augmentation,
        drug_swap_prob=drug_swap_prob,
    )
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE.type == "cuda" else False,
        persistent_workers=True if NUM_WORKERS > 0 else False,
        collate_fn=collate_fn,
    )


def block_to_batch_with_mask(block, obs_mask):
    """把单个 block + 当前观测 mask 打包成模型输入。"""
    y_flat = block["Y"].reshape(-1).astype(np.float32)
    obs_mask = np.asarray(obs_mask, dtype=np.float32)
    obs_values = y_flat * obs_mask
    loss_mask = block.get("loss_mask", np.ones(N_GRID, dtype=np.float32)).astype(np.float32)
    eval_dose_mask = block.get("eval_dose_mask", np.zeros(N_GRID, dtype=np.float32)).astype(np.float32)
    batch = {
        "block_id": torch.tensor([block["block_id"]], dtype=torch.long, device=DEVICE),
        "drugA_feat": torch.tensor(block["drugA_feat"][None, :], dtype=torch.float32, device=DEVICE),
        "drugB_feat": torch.tensor(block["drugB_feat"][None, :], dtype=torch.float32, device=DEVICE),
        "cell_feat": torch.tensor(block["cell_feat"][None, :], dtype=torch.float32, device=DEVICE),
        "grid_features": torch.tensor(block["grid_features"][None, :, :], dtype=torch.float32, device=DEVICE),
        "YA": torch.tensor(block["YA"][None, :], dtype=torch.float32, device=DEVICE),
        "YB": torch.tensor(block["YB"][None, :], dtype=torch.float32, device=DEVICE),
        "Y": torch.tensor(block["Y"][None, :, :], dtype=torch.float32, device=DEVICE),
        "Y_flat": torch.tensor(y_flat[None, :], dtype=torch.float32, device=DEVICE),
        "obs_mask": torch.tensor(obs_mask[None, :], dtype=torch.float32, device=DEVICE),
        "obs_values": torch.tensor(obs_values[None, :], dtype=torch.float32, device=DEVICE),
        "loss_mask": torch.tensor(loss_mask[None, :], dtype=torch.float32, device=DEVICE),
        "eval_dose_mask": torch.tensor(eval_dose_mask[None, :], dtype=torch.float32, device=DEVICE),
        "observed_k": torch.tensor([int(obs_mask.sum())], dtype=torch.long, device=DEVICE),
    }
    return batch


@torch.no_grad()
def ensemble_predict_flat(models, block, obs_mask):
    """返回 ensemble mean/std，用于主动选择与评估。"""
    preds = []
    batch = block_to_batch_with_mask(block, obs_mask)
    for model in models:
        model.eval()
        out = model(batch)
        pred = out["pred_flat"][0].detach().cpu().numpy().astype(np.float32)
        preds.append(pred)
    preds = np.stack(preds, axis=0)  # [E, 16]
    pred_mean = preds.mean(axis=0)
    pred_std = preds.std(axis=0)
    return pred_mean, pred_std, preds



@torch.no_grad()
def ensemble_predict_components_flat(models, block, obs_mask):
    """
    返回 active selection 需要的 ensemble 预测分量。

    pred_*    : 完整剂量响应矩阵预测。
    epsilon_* : 模型残差交互项 epsilon，来自 Pred = Base + epsilon。
    synergy_* : 基于 Bliss-like 单药预期的预测协同近似，synergy = pred - bliss_like。

    epsilon/base/synergy-related components are retained only for downstream diagnostics
    and do not participate in uncertainty acquisition.
    """
    preds = []
    epsilons = []
    bases = []
    batch = block_to_batch_with_mask(block, obs_mask)
    for model in models:
        model.eval()
        out = model(batch)
        preds.append(out["pred_flat"][0].detach().cpu().numpy().astype(np.float32))
        epsilons.append(out["epsilon_flat"][0].detach().cpu().numpy().astype(np.float32))
        bases.append(out["base_surface"][0].detach().cpu().numpy().reshape(-1).astype(np.float32))

    preds = np.stack(preds, axis=0)
    epsilons = np.stack(epsilons, axis=0)
    bases = np.stack(bases, axis=0)

    pred_mean = preds.mean(axis=0)
    pred_std = preds.std(axis=0)
    epsilon_mean = epsilons.mean(axis=0)
    epsilon_std = epsilons.std(axis=0)
    base_mean = bases.mean(axis=0)
    base_std = bases.std(axis=0)

    bliss = np.asarray(block["grid_features"][:, 10], dtype=np.float32)
    synergy_preds = preds - bliss[None, :]
    synergy_mean = synergy_preds.mean(axis=0)
    synergy_std = synergy_preds.std(axis=0)

    return {
        "pred_mean": pred_mean,
        "pred_std": pred_std,
        "preds": preds,
        "epsilon_mean": epsilon_mean,
        "epsilon_std": epsilon_std,
        "epsilons": epsilons,
        "base_mean": base_mean,
        "base_std": base_std,
        "synergy_mean": synergy_mean,
        "synergy_std": synergy_std,
        "synergy_preds": synergy_preds,
        "bliss": bliss,
    }

def initial_observed_mask(mode, rng):
    """构造初始 seed 点。"""
    mask = np.zeros(N_GRID, dtype=np.float32)
    if mode is None or mode == "none":
        return mask
    if mode == "random2":
        idx = rng.choice(N_GRID, size=min(2, N_GRID), replace=False)
        mask[idx] = 1.0
        return mask
    if mode == "corners4":
        idx = [0, GRID_SIZE_B - 1, (GRID_SIZE_A - 1) * GRID_SIZE_B, N_GRID - 1]
        mask[idx] = 1.0
        return mask
    if mode == "center4":
        # 4x4 中心区域: (1,1),(1,2),(2,1),(2,2)
        idx = [1 * GRID_SIZE_B + 1, 1 * GRID_SIZE_B + 2, 2 * GRID_SIZE_B + 1, 2 * GRID_SIZE_B + 2]
        mask[idx] = 1.0
        return mask
    raise ValueError(f"Unknown ACTIVE_INITIAL_SEED_MODE: {mode}")











def surface_shape_diagnostics(y_true_surface, y_pred_surface):
    """
    曲面形态诊断指标，用于评估补全矩阵拟合成曲面时是否保留相邻剂量变化趋势和局部弯曲形态。
    这些指标仍然基于 4x4 离散矩阵，不等同于真实连续剂量曲面验证。
    """
    yt = np.asarray(y_true_surface, dtype=float).reshape(GRID_SIZE_A, GRID_SIZE_B)
    yp = np.asarray(y_pred_surface, dtype=float).reshape(GRID_SIZE_A, GRID_SIZE_B)

    true_grad = []
    pred_grad = []
    true_grad.extend((yt[1:, :] - yt[:-1, :]).reshape(-1).tolist())
    true_grad.extend((yt[:, 1:] - yt[:, :-1]).reshape(-1).tolist())
    pred_grad.extend((yp[1:, :] - yp[:-1, :]).reshape(-1).tolist())
    pred_grad.extend((yp[:, 1:] - yp[:, :-1]).reshape(-1).tolist())
    true_grad = np.asarray(true_grad, dtype=float)
    pred_grad = np.asarray(pred_grad, dtype=float)

    grad_rmse = float(np.sqrt(mean_squared_error(true_grad, pred_grad))) if len(true_grad) > 0 else np.nan
    grad_mae = float(mean_absolute_error(true_grad, pred_grad)) if len(true_grad) > 0 else np.nan
    grad_corr = safe_pcc(true_grad, pred_grad)

    def laplacian_like(arr):
        vals = []
        for i in range(GRID_SIZE_A):
            for j in range(GRID_SIZE_B):
                neigh = []
                for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    ni, nj = i + di, j + dj
                    if 0 <= ni < GRID_SIZE_A and 0 <= nj < GRID_SIZE_B:
                        neigh.append(arr[ni, nj])
                if len(neigh) > 0:
                    vals.append(arr[i, j] - np.mean(neigh))
                else:
                    vals.append(0.0)
        return np.asarray(vals, dtype=float)

    true_lap = laplacian_like(yt)
    pred_lap = laplacian_like(yp)
    lap_rmse = float(np.sqrt(mean_squared_error(true_lap, pred_lap)))
    lap_mae = float(mean_absolute_error(true_lap, pred_lap))
    lap_corr = safe_pcc(true_lap, pred_lap)

    return {
        "surface_gradient_rmse": grad_rmse,
        "surface_gradient_mae": grad_mae,
        "surface_gradient_corr": grad_corr,
        "surface_laplacian_rmse": lap_rmse,
        "surface_laplacian_mae": lap_mae,
        "surface_laplacian_corr": lap_corr,
    }



def select_next_points(
    strategy,
    pred_mean,
    pred_std,
    obs_mask,
    batch_size,
    rng,
    epsilon_mean=None,
    epsilon_std=None,
    synergy_mean=None,
    synergy_std=None,
):
    """Uncertainty-only acquisition: select unobserved positions with largest ensemble std."""
    if strategy != "uncertainty":
        raise ValueError(
            f"This notebook is uncertainty-only, but got strategy={strategy!r}. "
            "Set ACTIVE_STRATEGIES = ['uncertainty']."
        )

    unobs_idx = np.where(obs_mask < 0.5)[0]
    if len(unobs_idx) == 0:
        return []

    batch_size = min(int(batch_size), len(unobs_idx))
    score = np.asarray(pred_std, dtype=float).copy()
    score[obs_mask > 0.5] = -np.inf
    return np.argsort(-score)[:batch_size].tolist()


def true_top_indices(y_flat, k=3):
    if MAXIMIZE_RESPONSE:
        return np.argsort(-y_flat)[:k]
    return np.argsort(y_flat)[:k]


def flat_to_ij(idx):
    return int(idx // GRID_SIZE_B), int(idx % GRID_SIZE_B)


def _top_region_metrics(true_score, pred_score, obs_mask, prefix, larger_is_better=True, topk=3):
    """用于高响应区域/协同热点区域的通用 top-k 评估。"""
    true_score = np.asarray(true_score, dtype=float)
    pred_score = np.asarray(pred_score, dtype=float)
    obs_mask = np.asarray(obs_mask, dtype=np.float32)

    if larger_is_better:
        true_order = np.argsort(-true_score)
        pred_order = np.argsort(-pred_score)
    else:
        true_order = np.argsort(true_score)
        pred_order = np.argsort(pred_score)

    true_best = int(true_order[0])
    pred_best = int(pred_order[0])
    true_topk = set(true_order[:topk].tolist())
    pred_topk = set(pred_order[:topk].tolist())

    ti, tj = flat_to_ij(true_best)
    pi, pj = flat_to_ij(pred_best)
    true_best_value = float(true_score[true_best])
    pred_best_true_value = float(true_score[pred_best])
    regret = float(true_best_value - pred_best_true_value) if larger_is_better else float(pred_best_true_value - true_best_value)

    observed_idx = np.where(obs_mask > 0.5)[0]
    if len(observed_idx) > 0:
        observed_set = set(observed_idx.tolist())
        observed_top1_hit = float(true_best in observed_set)
        observed_top3_hit = float(len(true_topk & observed_set) > 0)
        if larger_is_better:
            obs_best_idx = int(observed_idx[np.argmax(true_score[observed_idx])])
            obs_best_value = float(np.max(true_score[observed_idx]))
            observed_regret = float(true_best_value - obs_best_value)
        else:
            obs_best_idx = int(observed_idx[np.argmin(true_score[observed_idx])])
            obs_best_value = float(np.min(true_score[observed_idx]))
            observed_regret = float(obs_best_value - true_best_value)
        oi, oj = flat_to_ij(obs_best_idx)
        observed_best_distance = float(abs(ti - oi) + abs(tj - oj))
    else:
        observed_top1_hit = 0.0
        observed_top3_hit = 0.0
        obs_best_idx = -1
        obs_best_value = np.nan
        observed_regret = np.nan
        observed_best_distance = np.nan

    # top-25% 区域重叠：4x4 矩阵默认取前 4 个点。
    q = max(1, int(np.ceil(0.25 * len(true_score))))
    true_hot = set(true_order[:q].tolist())
    pred_hot = set(pred_order[:q].tolist())
    obs_hot = set(observed_idx.tolist()) if len(observed_idx) > 0 else set()
    pred_jaccard = float(len(true_hot & pred_hot) / max(1, len(true_hot | pred_hot)))
    observed_recall = float(len(true_hot & obs_hot) / max(1, len(true_hot)))

    return {
        f"{prefix}_pred_top1_hit": float(pred_best == true_best),
        f"{prefix}_pred_top3_hit": float(len(true_topk & pred_topk) > 0),
        f"{prefix}_pred_best_distance": float(abs(ti - pi) + abs(tj - pj)),
        f"{prefix}_pred_regret": regret,
        f"{prefix}_observed_top1_hit": observed_top1_hit,
        f"{prefix}_observed_top3_hit": observed_top3_hit,
        f"{prefix}_observed_best_distance": observed_best_distance,
        f"{prefix}_observed_regret": observed_regret,
        f"{prefix}_true_best_flat": true_best,
        f"{prefix}_pred_best_flat": pred_best,
        f"{prefix}_observed_best_flat": obs_best_idx,
        f"{prefix}_true_best_value": true_best_value,
        f"{prefix}_pred_best_true_value": pred_best_true_value,
        f"{prefix}_observed_best_true_value": obs_best_value,
        f"{prefix}_top25_jaccard_pred": pred_jaccard,
        f"{prefix}_top25_recall_observed": observed_recall,
    }


def evaluate_active_state(
    block,
    pred_mean,
    pred_std,
    obs_mask,
    epsilon_mean=None,
    epsilon_std=None,
    synergy_mean=None,
    synergy_std=None,
):
    """评估当前 active state：矩阵恢复、高响应区域覆盖、协同热点区域覆盖。"""
    y_flat = block["Y"].reshape(-1).astype(float)
    pred_flat = np.asarray(pred_mean, dtype=float)
    pred_std = np.asarray(pred_std, dtype=float)
    obs_mask = np.asarray(obs_mask, dtype=np.float32)
    miss_mask = 1.0 - obs_mask

    # 曲面级指标
    rmse_all = float(np.sqrt(mean_squared_error(y_flat, pred_flat)))
    mae_all = float(mean_absolute_error(y_flat, pred_flat))
    pcc_all = safe_pcc(y_flat, pred_flat)
    cindex_all = concordance_index(y_flat, pred_flat)

    if miss_mask.sum() > 0:
        miss_bool = miss_mask.astype(bool)
        rmse_missing = float(np.sqrt(mean_squared_error(y_flat[miss_bool], pred_flat[miss_bool])))
        mae_missing = float(mean_absolute_error(y_flat[miss_bool], pred_flat[miss_bool]))
        pcc_missing = safe_pcc(y_flat[miss_bool], pred_flat[miss_bool])
    else:
        rmse_missing = np.nan
        mae_missing = np.nan
        pcc_missing = np.nan

    # 高响应区域指标：沿用原 pred_top/observed_top 命名，方便和旧结果兼容。
    response_metrics = _top_region_metrics(
        true_score=y_flat,
        pred_score=pred_flat,
        obs_mask=obs_mask,
        prefix="response",
        larger_is_better=MAXIMIZE_RESPONSE,
        topk=3,
    )

    # 旧列名兼容。
    pred_top1_hit = response_metrics["response_pred_top1_hit"]
    pred_top3_hit = response_metrics["response_pred_top3_hit"]
    pred_best_distance = response_metrics["response_pred_best_distance"]
    pred_regret = response_metrics["response_pred_regret"]
    observed_top1_hit = response_metrics["response_observed_top1_hit"]
    observed_top3_hit = response_metrics["response_observed_top3_hit"]
    observed_best_distance = response_metrics["response_observed_best_distance"]
    observed_regret = response_metrics["response_observed_regret"]
    true_best = response_metrics["response_true_best_flat"]
    pred_best = response_metrics["response_pred_best_flat"]
    obs_best_idx = response_metrics["response_observed_best_flat"]
    true_best_value = response_metrics["response_true_best_value"]
    pred_best_true_value = response_metrics["response_pred_best_true_value"]
    obs_best_value = response_metrics["response_observed_best_true_value"]

    pred_surface = pred_flat.reshape(GRID_SIZE_A, GRID_SIZE_B)
    surface_shape_metrics = surface_shape_diagnostics(block["Y"], pred_surface)

    # 协同/残差热点指标：true synergy 用 Bliss-like 单药预期作为参照。
    bliss = np.asarray(block["grid_features"][:, 10], dtype=float)
    true_synergy = y_flat - bliss
    if synergy_mean is None:
        pred_synergy = pred_flat - bliss
    else:
        pred_synergy = np.asarray(synergy_mean, dtype=float)
    synergy_metrics = _top_region_metrics(
        true_score=true_synergy,
        pred_score=pred_synergy,
        obs_mask=obs_mask,
        prefix="synergy",
        larger_is_better=True,
        topk=3,
    )

    # residual/evidence summary
    epsilon_mean_arr = np.asarray(epsilon_mean, dtype=float) if epsilon_mean is not None else np.full_like(pred_flat, np.nan, dtype=float)
    epsilon_std_arr = np.asarray(epsilon_std, dtype=float) if epsilon_std is not None else np.full_like(pred_flat, np.nan, dtype=float)
    synergy_std_arr = np.asarray(synergy_std, dtype=float) if synergy_std is not None else np.full_like(pred_flat, np.nan, dtype=float)

    if miss_mask.sum() > 0:
        miss_bool = miss_mask.astype(bool)
        mean_epsilon_unobserved = float(np.nanmean(epsilon_mean_arr[miss_bool]))
        mean_positive_epsilon_unobserved = float(np.nanmean(np.maximum(epsilon_mean_arr[miss_bool], 0.0)))
        mean_abs_epsilon_unobserved = float(np.nanmean(np.abs(epsilon_mean_arr[miss_bool])))
        mean_synergy_unobserved = float(np.nanmean(pred_synergy[miss_bool]))
        mean_positive_synergy_unobserved = float(np.nanmean(np.maximum(pred_synergy[miss_bool], 0.0)))
        mean_synergy_uncertainty_unobserved = float(np.nanmean(synergy_std_arr[miss_bool]))
    else:
        mean_epsilon_unobserved = 0.0
        mean_positive_epsilon_unobserved = 0.0
        mean_abs_epsilon_unobserved = 0.0
        mean_synergy_unobserved = 0.0
        mean_positive_synergy_unobserved = 0.0
        mean_synergy_uncertainty_unobserved = 0.0

    # dose-point cold-start 专用指标：本版只跑 block_random_split，保留字段兼容旧代码。
    eval_dose_mask = np.asarray(block.get("eval_dose_mask", np.zeros(N_GRID, dtype=np.float32)), dtype=np.float32)
    if eval_dose_mask.sum() > 0:
        held_bool = eval_dose_mask.astype(bool)
        rmse_heldout_dose = float(np.sqrt(mean_squared_error(y_flat[held_bool], pred_flat[held_bool])))
        mae_heldout_dose = float(mean_absolute_error(y_flat[held_bool], pred_flat[held_bool]))
        pcc_heldout_dose = safe_pcc(y_flat[held_bool], pred_flat[held_bool])
    else:
        rmse_heldout_dose = np.nan
        mae_heldout_dose = np.nan
        pcc_heldout_dose = np.nan

    out = {
        "rmse_all": rmse_all,
        "mae_all": mae_all,
        "pcc_all": pcc_all,
        "c_index_all": cindex_all,
        "rmse_missing": rmse_missing,
        "mae_missing": mae_missing,
        "pcc_missing": pcc_missing,
        "adjacent_delta_corr": adjacent_delta_corr(block["Y"], pred_surface),
        "pred_top1_hit": pred_top1_hit,
        "pred_top3_hit": pred_top3_hit,
        "pred_best_distance": pred_best_distance,
        "pred_regret": pred_regret,
        "observed_top1_hit": observed_top1_hit,
        "observed_top3_hit": observed_top3_hit,
        "observed_best_distance": observed_best_distance,
        "observed_regret": observed_regret,
        "true_best_flat": true_best,
        "pred_best_flat": pred_best,
        "observed_best_flat": obs_best_idx,
        "true_best_value": true_best_value,
        "pred_best_true_value": pred_best_true_value,
        "observed_best_true_value": obs_best_value,
        "mean_uncertainty_unobserved": float(np.mean(pred_std[miss_mask.astype(bool)])) if miss_mask.sum() > 0 else 0.0,
        "mean_epsilon_unobserved": mean_epsilon_unobserved,
        "mean_positive_epsilon_unobserved": mean_positive_epsilon_unobserved,
        "mean_abs_epsilon_unobserved": mean_abs_epsilon_unobserved,
        "mean_synergy_unobserved": mean_synergy_unobserved,
        "mean_positive_synergy_unobserved": mean_positive_synergy_unobserved,
        "mean_synergy_uncertainty_unobserved": mean_synergy_uncertainty_unobserved,
        "rmse_heldout_dose": rmse_heldout_dose,
        "mae_heldout_dose": mae_heldout_dose,
        "pcc_heldout_dose": pcc_heldout_dose,
    }
    out.update(response_metrics)
    out.update(synergy_metrics)
    out.update(surface_shape_metrics)
    return out

def simulate_active_one_block(models, block, strategy, repeat_seed, total_budgets):
    """
    在单个 block 上模拟主动剂量点选择。
    注意：这里的“更新模型”是 amortized update：揭示新点后更新 obs_mask/obs_values，
    再重新前向传播得到新的曲面预测和不确定性；不在每个测试 block 上重新训练网络参数。
    """
    rng = np.random.default_rng(repeat_seed)
    y_flat = block["Y"].reshape(-1).astype(np.float32)

    obs_mask = initial_observed_mask(ACTIVE_INITIAL_SEED_MODE, rng)
    selected_history = np.where(obs_mask > 0.5)[0].tolist()
    rows = []
    traj_rows = []

    max_budget = int(max(total_budgets))
    round_id = 0

    while int(obs_mask.sum()) < max_budget:
        comp = ensemble_predict_components_flat(models, block, obs_mask)
        need = min(ACTIVE_BATCH_SIZE, max_budget - int(obs_mask.sum()))
        next_points = select_next_points(
            strategy=strategy,
            pred_mean=comp["pred_mean"],
            pred_std=comp["pred_std"],
            obs_mask=obs_mask,
            batch_size=need,
            rng=rng,
            epsilon_mean=comp["epsilon_mean"],
            epsilon_std=comp["epsilon_std"],
            synergy_mean=comp["synergy_mean"],
            synergy_std=comp["synergy_std"],
        )
        if len(next_points) == 0:
            break

        round_id += 1
        for idx in next_points:
            obs_mask[idx] = 1.0
            selected_history.append(int(idx))
            if SAVE_ACTIVE_TRAJECTORY:
                i, j = flat_to_ij(idx)
                traj_rows.append({
                    "block_id": block["block_id"],
                    "drugA_name": block["drugA_name"],
                    "drugB_name": block["drugB_name"],
                    "cell_line": block["cell_line"],
                    "strategy": strategy,
                    "repeat_seed": repeat_seed,
                    "round": round_id,
                    "selected_flat": int(idx),
                    "selected_i": i,
                    "selected_j": j,
                    "doseA": float(block["doseA_levels"][i]),
                    "doseB": float(block["doseB_levels"][j]),
                    "true_response": float(y_flat[idx]),
                    "pred_response_before_measurement": float(comp["pred_mean"][idx]),
                    "pred_uncertainty_before_measurement": float(comp["pred_std"][idx]),
                    "pred_epsilon_before_measurement": float(comp["epsilon_mean"][idx]),
                    "pred_synergy_before_measurement": float(comp["synergy_mean"][idx]),
                    "true_bliss_expected": float(comp["bliss"][idx]),
                    "true_bliss_synergy": float(y_flat[idx] - comp["bliss"][idx]),
                    "observed_count_after": int(obs_mask.sum()),
                })

        current_budget = int(obs_mask.sum())
        if current_budget in total_budgets:
            comp = ensemble_predict_components_flat(models, block, obs_mask)
            met = evaluate_active_state(
                block=block,
                pred_mean=comp["pred_mean"],
                pred_std=comp["pred_std"],
                obs_mask=obs_mask,
                epsilon_mean=comp["epsilon_mean"],
                epsilon_std=comp["epsilon_std"],
                synergy_mean=comp["synergy_mean"],
                synergy_std=comp["synergy_std"],
            )
            row = {
                "block_id": block["block_id"],
                "drugA_name": block["drugA_name"],
                "drugB_name": block["drugB_name"],
                "cell_line": block["cell_line"],
                "strategy": strategy,
                "repeat_seed": repeat_seed,
                "budget": current_budget,
                "round": round_id,
                "initial_seed_mode": ACTIVE_INITIAL_SEED_MODE,
                "active_batch_size": ACTIVE_BATCH_SIZE,
                "selected_history": ";".join(map(str, selected_history)),
            }
            row.update(met)
            rows.append(row)

    # 如果初始 seed 本身就等于某些 budget，也补记一次。
    initial_k = int(initial_observed_mask(ACTIVE_INITIAL_SEED_MODE, np.random.default_rng(repeat_seed)).sum())
    recorded_budgets = set([r["budget"] for r in rows])
    if initial_k in total_budgets and initial_k not in recorded_budgets:
        init_mask = initial_observed_mask(ACTIVE_INITIAL_SEED_MODE, np.random.default_rng(repeat_seed))
        comp = ensemble_predict_components_flat(models, block, init_mask)
        met = evaluate_active_state(
            block=block,
            pred_mean=comp["pred_mean"],
            pred_std=comp["pred_std"],
            obs_mask=init_mask,
            epsilon_mean=comp["epsilon_mean"],
            epsilon_std=comp["epsilon_std"],
            synergy_mean=comp["synergy_mean"],
            synergy_std=comp["synergy_std"],
        )
        row = {
            "block_id": block["block_id"],
            "drugA_name": block["drugA_name"],
            "drugB_name": block["drugB_name"],
            "cell_line": block["cell_line"],
            "strategy": strategy,
            "repeat_seed": repeat_seed,
            "budget": initial_k,
            "round": 0,
            "initial_seed_mode": ACTIVE_INITIAL_SEED_MODE,
            "active_batch_size": ACTIVE_BATCH_SIZE,
            "selected_history": ";".join(map(str, np.where(init_mask > 0.5)[0].tolist())),
        }
        row.update(met)
        rows.insert(0, row)

    return rows, traj_rows

def run_active_simulation(models, test_blocks, setting_dir, setting_name):
    """在整个测试集上运行 retrospective active-learning simulation。"""
    active_rows = []
    traj_rows = []
    budgets = set(map(int, ACTIVE_TOTAL_BUDGETS))

    print("\n========== SAFIR-Active simulation ==========")
    print("Strategies:", ACTIVE_STRATEGIES)
    print("Initial seed:", ACTIVE_INITIAL_SEED_MODE)
    print("Batch size:", ACTIVE_BATCH_SIZE)
    print("Budgets:", ACTIVE_TOTAL_BUDGETS)
    print("Repeats:", ACTIVE_REPEATS)

    for strategy in ACTIVE_STRATEGIES:
        print(f"Running active strategy: {strategy}")
        for r in range(ACTIVE_REPEATS):
            # uncertainty-only：不同 repeat 使用不同 random2 初始观测，
            # 同一 repeat 内每个 block 的初始随机种子可复现。
            repeat_seed_base = RANDOM_STATE + 10000 + r * 100000
            for bi, block in enumerate(test_blocks):
                paired_seed = repeat_seed_base + bi
                rows, traj = simulate_active_one_block(
                    models=models,
                    block=block,
                    strategy=strategy,
                    repeat_seed=paired_seed,
                    total_budgets=budgets,
                )
                active_rows.extend(rows)
                traj_rows.extend(traj)

    active_df = pd.DataFrame(active_rows)
    active_path = setting_dir / f"{setting_name}_active_dose_selection_metrics.csv"
    active_path.parent.mkdir(parents=True, exist_ok=True)
    active_df.to_csv(active_path, index=False, encoding="utf-8-sig")

    if SAVE_ACTIVE_TRAJECTORY:
        traj_df = pd.DataFrame(traj_rows)
        traj_path = setting_dir / f"{setting_name}_active_dose_selection_trajectory.csv"
        traj_path.parent.mkdir(parents=True, exist_ok=True)
        traj_df.to_csv(traj_path, index=False, encoding="utf-8-sig")
    else:
        traj_df = None

    summary_df = summarize_active_metrics(active_df)
    summary_path = setting_dir / f"{setting_name}_active_dose_selection_summary.csv"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print("Saved active metrics:", active_path)
    print("Saved active summary:", summary_path)
    print(summary_df)
    return active_df, summary_df, traj_df


def summarize_active_metrics(active_df):
    group_cols = ["strategy", "budget"]
    metric_cols = [
        # 矩阵补全质量
        "rmse_all", "mae_all", "pcc_all", "c_index_all",
        "rmse_missing", "mae_missing", "pcc_missing", "adjacent_delta_corr",
        "surface_gradient_rmse", "surface_gradient_mae", "surface_gradient_corr",
        "surface_laplacian_rmse", "surface_laplacian_mae", "surface_laplacian_corr",
        "mean_uncertainty_unobserved",
        # 高响应区域发现
        "pred_top1_hit", "pred_top3_hit", "pred_best_distance", "pred_regret",
        "observed_top1_hit", "observed_top3_hit", "observed_best_distance", "observed_regret",
        "response_pred_top1_hit", "response_pred_top3_hit", "response_pred_best_distance", "response_pred_regret",
        "response_observed_top1_hit", "response_observed_top3_hit", "response_observed_best_distance", "response_observed_regret",
        "response_top25_jaccard_pred", "response_top25_recall_observed",
        # 残差/协同热点发现
        "synergy_pred_top1_hit", "synergy_pred_top3_hit", "synergy_pred_best_distance", "synergy_pred_regret",
        "synergy_observed_top1_hit", "synergy_observed_top3_hit", "synergy_observed_best_distance", "synergy_observed_regret",
        "synergy_top25_jaccard_pred", "synergy_top25_recall_observed",
        "mean_epsilon_unobserved", "mean_positive_epsilon_unobserved", "mean_abs_epsilon_unobserved",
        "mean_synergy_unobserved", "mean_positive_synergy_unobserved", "mean_synergy_uncertainty_unobserved",
        # 兼容旧 dose-point cold-start 字段，本版 block_random_split 下为 NaN。
        "rmse_heldout_dose", "mae_heldout_dose", "pcc_heldout_dose",
    ]
    metric_cols = [c for c in metric_cols if c in active_df.columns]
    agg = active_df.groupby(group_cols)[metric_cols].mean().reset_index()
    agg = agg.sort_values(["budget", "strategy"]).reset_index(drop=True)
    return agg

@torch.no_grad()
def predict_blocks_with_mask_ensemble(models, blocks, observed_k=0, mask_repeat=1, seed=42):
    """固定 K 的 ensemble 评估，用于保留原先 K=0/2/... 的补全表。"""
    rng = np.random.default_rng(seed)
    rows = []
    for repeat in range(mask_repeat):
        for b in blocks:
            y_flat = b["Y"].reshape(-1).astype(np.float32)
            obs_mask = make_observed_mask(N_GRID, observed_k, rng)
            pred_mean, pred_std, _ = ensemble_predict_flat(models, b, obs_mask)
            met = evaluate_active_state(b, pred_mean, pred_std, obs_mask)
            row = {
                "repeat": repeat,
                "block_id": b["block_id"],
                "drugA_name": b["drugA_name"],
                "drugB_name": b["drugB_name"],
                "cell_line": b["cell_line"],
                "observed_k": observed_k,
            }
            # 兼容原 summary 字段名
            row.update({
                "rmse_all": met["rmse_all"],
                "mae_all": met["mae_all"],
                "pcc_all": met["pcc_all"],
                "r2_all": float(r2_score(y_flat, pred_mean)) if np.std(y_flat) > 1e-12 else np.nan,
                "c_index_all": met["c_index_all"],
                "rmse_missing": met["rmse_missing"],
                "mae_missing": met["mae_missing"],
                "pcc_missing": met["pcc_missing"],
                "c_index_missing": np.nan,
                "adjacent_delta_corr": met["adjacent_delta_corr"],
                "top1_hit": met["pred_top1_hit"],
                "top3_hit": met["pred_top3_hit"],
                "best_dose_distance": met["pred_best_distance"],
                "true_best_flat": met["true_best_flat"],
                "pred_best_flat": met["pred_best_flat"],
                "mean_uncertainty_unobserved": met["mean_uncertainty_unobserved"],
                "rmse_heldout_dose": met.get("rmse_heldout_dose", np.nan),
                "mae_heldout_dose": met.get("mae_heldout_dose", np.nan),
                "pcc_heldout_dose": met.get("pcc_heldout_dose", np.nan),
            })
            rows.append(row)
    return pd.DataFrame(rows)


def train_single_model_for_ensemble(tr_blocks, val_blocks, setting_dir, model_mode_config, seed, ensemble_id):
    """训练一个 SAFIR-Surface 预测引擎，作为 ensemble 成员。"""
    seed_everything(seed)
    train_loader = make_loader_with_seed(
        tr_blocks,
        TRAIN_OBSERVED_K_LIST,
        training=True,
        shuffle=True,
        seed=seed,
        drug_swap_augmentation=DRUG_SWAP_AUGMENTATION,
        drug_swap_prob=DRUG_SWAP_PROB,
    )
    # Validation keeps the original A/B orientation. It still samples Random Mask-K
    # exactly as before, but does NOT receive drug-swap augmentation.
    val_loader = make_loader_with_seed(
        val_blocks,
        TRAIN_OBSERVED_K_LIST,
        training=True,
        shuffle=False,
        seed=seed + 17,
        drug_swap_augmentation=False,
        drug_swap_prob=0.0,
    )

    drug_dim = tr_blocks[0]["drugA_feat"].shape[0]
    cell_dim = tr_blocks[0]["cell_feat"].shape[0]
    grid_feat_dim = tr_blocks[0]["grid_features"].shape[1]

    model = SAFIRDoseSurfaceCompletionNet(
        drug_dim=drug_dim,
        cell_dim=cell_dim,
        grid_feat_dim=grid_feat_dim,
        use_grid_gnn=model_mode_config["use_grid_gnn"],
        use_local_obs=model_mode_config["use_local_obs"],
        use_residual_anchor=model_mode_config["use_residual_anchor"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
    )

    best_val = float("inf")
    best_epoch = None
    best_state = None
    epochs_no_improve = 0
    history = []

    print(f"\n--- Train ensemble member {ensemble_id} | seed={seed} ---")

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        epoch_losses = []
        for batch in train_loader:
            batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(batch)
            loss, loss_dict = compute_loss(out, batch)
            loss.backward()
            if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            epoch_losses.append(loss_dict)

        train_loss = float(np.mean([x["loss_total"] for x in epoch_losses]))
        train_recon = float(np.mean([x["loss_reconstruction"] for x in epoch_losses]))
        train_missing = float(np.mean([x["loss_missing"] for x in epoch_losses]))
        train_obs = float(np.mean([x["loss_obs"] for x in epoch_losses]))
        train_delta = float(np.mean([x["loss_delta"] for x in epoch_losses]))
        train_residual = float(np.mean([x["loss_residual_consistency"] for x in epoch_losses]))
        val_loss = evaluate_val_loss(model, val_loader)
        scheduler.step(val_loss)

        history.append({
            "ensemble_id": ensemble_id,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_reconstruction_loss": train_recon,
            "train_missing_loss": train_missing,
            "train_observed_loss": train_obs,
            "train_delta_loss": train_delta,
            "train_residual_consistency_loss": train_residual,
            "val_loss": val_loss,
            "lr": optimizer.param_groups[0]["lr"],
        })

        print(
            f"Ensemble {ensemble_id} | Epoch {epoch:03d}/{NUM_EPOCHS} | "
            f"Train {train_loss:.5f} | Recon {train_recon:.5f} | "
            f"Delta {train_delta:.5f} | Residual {train_residual:.5f} | Val {val_loss:.5f}"
        )

        if val_loss < best_val - MIN_DELTA:
            best_val = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(best_state, setting_dir / f"best_model_ensemble{ensemble_id}_seed{seed}.pt")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping ensemble {ensemble_id} at epoch {epoch}. Best epoch={best_epoch}, val={best_val:.6f}")
            break

    pd.DataFrame(history).to_csv(
        setting_dir / f"training_history_ensemble{ensemble_id}_seed{seed}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, {"ensemble_id": ensemble_id, "seed": seed, "best_epoch": best_epoch, "best_val_loss": best_val}


def train_one_setting_active(train_blocks, val_blocks, test_blocks, setting_name, model_mode_name, model_mode_config):
    """训练 ensemble 预测引擎，并进行固定 K 评估 + 主动剂量点选择模拟。"""
    setting_dir = RESULT_DIR / setting_name
    setting_dir.mkdir(parents=True, exist_ok=True)

    # Train/Val/Test 已在 CRISPR 特征拟合之前固定；这里禁止重新划分。
    tr_blocks = list(train_blocks)
    val_blocks = list(val_blocks)

    print(f"\n========== Start SAFIR-Active training: {setting_name} ==========")
    print("Train blocks:", len(tr_blocks))
    print("Val blocks:", len(val_blocks))
    print("Test blocks:", len(test_blocks))
    print("Train observed K list:", TRAIN_OBSERVED_K_LIST)
    print("Drug-swap augmentation:", DRUG_SWAP_AUGMENTATION, "prob=", DRUG_SWAP_PROB)
    print("Active ensemble size:", ACTIVE_ENSEMBLE_SIZE)

    models = []
    ensemble_info = []
    seeds = ACTIVE_ENSEMBLE_SEEDS[:ACTIVE_ENSEMBLE_SIZE]
    for eid, seed in enumerate(seeds):
        model, info = train_single_model_for_ensemble(
            tr_blocks=tr_blocks,
            val_blocks=val_blocks,
            setting_dir=setting_dir,
            model_mode_config=model_mode_config,
            seed=int(seed),
            ensemble_id=eid,
        )
        models.append(model)
        ensemble_info.append(info)

    pd.DataFrame(ensemble_info).to_csv(setting_dir / "ensemble_training_summary.csv", index=False, encoding="utf-8-sig")

    # 固定 K 评估：保留原 SAFIR-Surface 补全结果，用 ensemble mean 做预测。
    fixed_k_metrics = []
    for k in EVAL_OBSERVED_K_LIST:
        repeat = EVAL_MASK_REPEATS if k > 0 else 1
        mdf = predict_blocks_with_mask_ensemble(
            models=models,
            blocks=test_blocks,
            observed_k=k,
            mask_repeat=repeat,
            seed=RANDOM_STATE + 2000 + k,
        )
        fixed_k_metrics.append(mdf)
    fixed_k_metrics_df = pd.concat(fixed_k_metrics, axis=0, ignore_index=True)
    fixed_k_summary_df = summarize_metrics(fixed_k_metrics_df)
    fixed_k_metrics_df.to_csv(setting_dir / f"{setting_name}_ensemble_fixedK_metrics.csv", index=False, encoding="utf-8-sig")
    fixed_k_summary_df.to_csv(setting_dir / f"{setting_name}_ensemble_fixedK_summary.csv", index=False, encoding="utf-8-sig")

    # 主动剂量点选择模拟：这是升级后的核心实验。
    if ACTIVE_ENABLE:
        active_df, active_summary_df, traj_df = run_active_simulation(
            models=models,
            test_blocks=test_blocks,
            setting_dir=setting_dir,
            setting_name=setting_name,
        )
    else:
        active_df = pd.DataFrame()
        active_summary_df = pd.DataFrame()

    config = {
        "setting_name": setting_name,
        "model_mode_name": model_mode_name,
        "model_mode_desc": model_mode_config.get("desc", ""),
        "use_grid_gnn": model_mode_config["use_grid_gnn"],
        "use_local_obs": model_mode_config["use_local_obs"],
        "use_residual_anchor": model_mode_config["use_residual_anchor"],
        "cell_feature_type": model_mode_config["cell_feature_type"],
        "main_experiment_type": model_mode_config.get("main_experiment_type", "unknown"),
        "active_cell_feature_file": str(ACTIVE_CELL_FEATURE_FILE),
        "split_reference_file": str(SPLIT_REFERENCE_FILE),
        "crispr_preprocessor_file": str(CRISPR_PREPROCESSOR_FILE),
        "leakage_control": "split first; Top1000/median/scaler fit on train cells only; val/test transform only; test excluded from model selection",
        "train_blocks": len(tr_blocks),
        "val_blocks": len(val_blocks),
        "test_blocks": len(test_blocks),
        "train_observed_k_list": TRAIN_OBSERVED_K_LIST,
        "eval_observed_k_list": EVAL_OBSERVED_K_LIST,
        "global_fusion": "ordered_concat_[hA,hB,hA*hB,abs(hA-hB),hC]",
        "drug_swap_augmentation": DRUG_SWAP_AUGMENTATION,
        "drug_swap_probability": DRUG_SWAP_PROB,
        "drug_swap_scope": "training_only; validation/test/fixed-K/active evaluation remain original orientation",
        "active_enable": ACTIVE_ENABLE,
        "active_ensemble_size": ACTIVE_ENSEMBLE_SIZE,
        "active_ensemble_seeds": seeds,
        "active_initial_seed_mode": ACTIVE_INITIAL_SEED_MODE,
        "active_batch_size": ACTIVE_BATCH_SIZE,
        "active_total_budgets": ACTIVE_TOTAL_BUDGETS,
        "active_repeats": ACTIVE_REPEATS,
        "active_strategies": ACTIVE_STRATEGIES,
        "maximize_response": MAXIMIZE_RESPONSE,
        "num_epochs_max": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "lambda_obs_recon": LAMBDA_OBS_RECON,
        "lambda_delta": LAMBDA_DELTA,
        "lambda_residual_consistency": LAMBDA_RESIDUAL_CONSISTENCY,
        "loss_objective": "reconstruction + lambda_delta*delta + lambda_residual*residual_consistency",
        "model_name": "SAFIR-Active-OrderedFusion-DrugSwapAug-UncertaintyOnly-SimplifiedLoss",
        "important_note": "Active update means updating obs_mask/obs_values with newly revealed dose points and recomputing ensemble predictions; network parameters are not refit inside each test block.",
    }
    with open(setting_dir / "run_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    print(f"\n========== Finished SAFIR-Active setting: {setting_name} ==========")
    print("Fixed-K summary:")
    print(fixed_k_summary_df)
    if ACTIVE_ENABLE:
        print("Active summary:")
        print(active_summary_df)

    return models, fixed_k_summary_df, fixed_k_metrics_df, active_summary_df, active_df


In [15]:
# ============================================================
# 12. Main
# ============================================================


def run_one_model_mode(model_mode_name):
    global RESULT_DIR, ACTIVE_CELL_FEATURE_FILE, TRAIN_OBSERVED_K_LIST, ACTIVE_STRATEGIES

    if model_mode_name not in MODEL_CONFIGS:
        raise ValueError(f"Unknown model_mode mode: {model_mode_name}. Available: {list(MODEL_CONFIGS)}")

    model_mode_config = MODEL_CONFIGS[model_mode_name]

    RESULT_DIR = RESULT_ROOT_DIR / model_mode_name
    RESULT_DIR.mkdir(parents=True, exist_ok=True)

    # 每个主实验允许覆盖训练 mask-K 列表和 active 策略。
    # 注意：这里从 DEFAULT_TRAIN_OBSERVED_K_LIST 回退，避免上一个模型模式的全局 TRAIN_OBSERVED_K_LIST 污染下一个实验。
    TRAIN_OBSERVED_K_LIST = list(model_mode_config.get("train_observed_k_list", DEFAULT_TRAIN_OBSERVED_K_LIST))
    ACTIVE_STRATEGIES = list(model_mode_config.get("active_strategies", ["uncertainty"]))

    if model_mode_config["cell_feature_type"] == "Expression_977":
        ACTIVE_CELL_FEATURE_FILE = EXPRESSION_CELL_FEATURE_FILE
    else:
        ACTIVE_CELL_FEATURE_FILE = CELL_FEATURE_FILE

    print("\n" + "=" * 80)
    print(f"Start model_mode: {model_mode_name}")
    print("Description:", model_mode_config["desc"])
    print("Active cell feature file:", ACTIVE_CELL_FEATURE_FILE)
    print("Model result dir:", RESULT_DIR)
    print("Train observed K list for this experiment:", TRAIN_OBSERVED_K_LIST)
    print("Active strategies for this experiment:", ACTIVE_STRATEGIES)
    print("=" * 80)

    for _label, _path in {
        "Train-only CRISPR cell features": CELL_FEATURE_FILE,
        "Fixed split reference": SPLIT_REFERENCE_FILE,
        "CRISPR preprocessor audit": CRISPR_PREPROCESSOR_FILE,
    }.items():
        if not Path(_path).exists():
            raise FileNotFoundError(
                f"{_label} 不存在：{_path}\n"
                "请先完整运行 01_ONEIL_CRISPR_Top1000_TrainOnly_NoLeakage_v2.ipynb。"
            )

    combo_df, drug_features, cell_features = load_raw_data()
    point_df = preprocess_combo_points(combo_df, drug_features, cell_features)
    blocks, block_summary_df = build_surface_blocks(
        point_df=point_df,
        drug_features=drug_features,
        cell_features=cell_features,
    )

    fixed_summaries = []
    active_summaries = []

    if "block_random_split" in RUN_SETTINGS:
        random_train_blocks, random_val_blocks, random_test_blocks = load_fixed_no_leakage_block_split(
            blocks, SPLIT_REFERENCE_FILE
        )
        _, fixed_summary, _, active_summary, _ = train_one_setting_active(
            random_train_blocks,
            random_val_blocks,
            random_test_blocks,
            setting_name="block_random_split",
            model_mode_name=model_mode_name,
            model_mode_config=model_mode_config,
        )
        fixed_summary = fixed_summary.copy(); fixed_summary["setting"] = "block_random_split"
        active_summary = active_summary.copy(); active_summary["setting"] = "block_random_split"
        fixed_summaries.append(fixed_summary); active_summaries.append(active_summary)

    if "cell_line_cold_start" in RUN_SETTINGS:
        raise RuntimeError("cell_line_cold_start 需要针对该 cold-start split 重新拟合 CRISPR Top1000；不能复用 block-random 的预处理器。")
        cold_train_blocks, cold_test_blocks = make_cell_line_cold_start_split(blocks)
        _, fixed_summary, _, active_summary, _ = train_one_setting_active(
            cold_train_blocks,
            cold_test_blocks,
            setting_name="cell_line_cold_start",
            model_mode_name=model_mode_name,
            model_mode_config=model_mode_config,
        )
        fixed_summary = fixed_summary.copy(); fixed_summary["setting"] = "cell_line_cold_start"
        active_summary = active_summary.copy(); active_summary["setting"] = "cell_line_cold_start"
        fixed_summaries.append(fixed_summary); active_summaries.append(active_summary)

    if "drug_combination_cold_start" in RUN_SETTINGS:
        raise RuntimeError("drug_combination_cold_start 需要针对该 cold-start split 重新拟合 CRISPR Top1000；不能复用 block-random 的预处理器。")
        combo_train_blocks, combo_test_blocks = make_drug_combination_cold_start_split(blocks)
        _, fixed_summary, _, active_summary, _ = train_one_setting_active(
            combo_train_blocks,
            combo_test_blocks,
            setting_name="drug_combination_cold_start",
            model_mode_name=model_mode_name,
            model_mode_config=model_mode_config,
        )
        fixed_summary = fixed_summary.copy(); fixed_summary["setting"] = "drug_combination_cold_start"
        active_summary = active_summary.copy(); active_summary["setting"] = "drug_combination_cold_start"
        fixed_summaries.append(fixed_summary); active_summaries.append(active_summary)

    if "single_drug_cold_start" in RUN_SETTINGS:
        raise RuntimeError("single_drug_cold_start 需要针对该 cold-start split 重新拟合 CRISPR Top1000；不能复用 block-random 的预处理器。")
        single_train_blocks, single_test_blocks = make_single_drug_cold_start_split(blocks)
        _, fixed_summary, _, active_summary, _ = train_one_setting_active(
            single_train_blocks,
            single_test_blocks,
            setting_name="single_drug_cold_start",
            model_mode_name=model_mode_name,
            model_mode_config=model_mode_config,
        )
        fixed_summary = fixed_summary.copy(); fixed_summary["setting"] = "single_drug_cold_start"
        active_summary = active_summary.copy(); active_summary["setting"] = "single_drug_cold_start"
        fixed_summaries.append(fixed_summary); active_summaries.append(active_summary)

    if "dose_point_cold_start" in RUN_SETTINGS:
        raise RuntimeError("dose_point_cold_start 需要针对该 cold-start split 重新拟合 CRISPR Top1000；不能复用 block-random 的预处理器。")
        dose_train_blocks, dose_test_blocks = make_dose_point_cold_start_split(blocks)
        _, fixed_summary, _, active_summary, _ = train_one_setting_active(
            dose_train_blocks,
            dose_test_blocks,
            setting_name="dose_point_cold_start",
            model_mode_name=model_mode_name,
            model_mode_config=model_mode_config,
        )
        fixed_summary = fixed_summary.copy(); fixed_summary["setting"] = "dose_point_cold_start"
        active_summary = active_summary.copy(); active_summary["setting"] = "dose_point_cold_start"
        fixed_summaries.append(fixed_summary); active_summaries.append(active_summary)

    if len(fixed_summaries) > 0:
        final_fixed_summary = pd.concat(fixed_summaries, axis=0, ignore_index=True)
        final_fixed_summary["model_mode"] = model_mode_name
        final_fixed_summary["main_experiment_type"] = model_mode_config.get("main_experiment_type", "unknown")
        final_fixed_summary["cell_feature_type"] = model_mode_config["cell_feature_type"]
        final_fixed_summary.to_csv(RESULT_DIR / f"final_fixedK_summary_{model_mode_name}.csv", index=False, encoding="utf-8-sig")
    else:
        final_fixed_summary = pd.DataFrame()

    if len(active_summaries) > 0:
        final_active_summary = pd.concat(active_summaries, axis=0, ignore_index=True)
        final_active_summary["model_mode"] = model_mode_name
        final_active_summary["main_experiment_type"] = model_mode_config.get("main_experiment_type", "unknown")
        final_active_summary["cell_feature_type"] = model_mode_config["cell_feature_type"]
        final_active_summary.to_csv(RESULT_DIR / f"final_active_summary_{model_mode_name}.csv", index=False, encoding="utf-8-sig")
    else:
        final_active_summary = pd.DataFrame()

    print("\n========== SAFIR-Active final fixed-K summary ==========")
    print(final_fixed_summary)
    print("\n========== SAFIR-Active final active-selection summary ==========")
    print(final_active_summary)
    print("\nModel results saved to:", RESULT_DIR)

    return final_fixed_summary, final_active_summary


def main():
    seed_everything(RANDOM_STATE)
    all_fixed = []
    all_active = []
    for model_mode_name in MODEL_MODES:
        fixed_summary, active_summary = run_one_model_mode(model_mode_name)
        all_fixed.append(fixed_summary)
        all_active.append(active_summary)

    if len(all_fixed) > 0:
        pd.concat(all_fixed, axis=0, ignore_index=True).to_csv(
            RESULT_ROOT_DIR / "final_fixedK_summary_all_modes.csv",
            index=False,
            encoding="utf-8-sig",
        )
    if len(all_active) > 0:
        pd.concat(all_active, axis=0, ignore_index=True).to_csv(
            RESULT_ROOT_DIR / "final_active_summary_all_modes.csv",
            index=False,
            encoding="utf-8-sig",
        )

    print("\n" + "=" * 80)
    print("SAFIR full No-GNN uncertainty-only simplified-loss experiment finished (no-leakage CRISPR).")
    print("All results saved to:", RESULT_ROOT_DIR)
    print("=" * 80)


if __name__ == "__main__":
    main()



Start model_mode: ordered_fusion_swapaug
Description: Full No-GNN SAFIR with original ordered global fusion + drug-swap training augmentation + residual anchor + CRISPR + Random Mask-K; simplified reconstruction/delta/residual-consistency loss; uncertainty-only active acquisition under fixed no-leakage 70/10/20 block split
Active cell feature file: D:\6.2\DecoDoseNet-main\oneil_cell_features_for_decodosenet\no_leakage_blocksplit_seed42\CRISPR_TopVar1000_TrainOnly_cell_features.csv
Model result dir: D:\6.2\DecoDoseNet-main\results\SAFIR_SwapAug_Unc_SimpleLoss_v2\ordered_fusion_swapaug
Train observed K list for this experiment: [0, 2, 4, 4, 6, 6, 8, 8, 10, 12]
Active strategies for this experiment: ['uncertainty']

========== Raw data ==========
Combo shape: (94224, 8)
Combo columns: ['drugA_name', 'drugA_conc', 'drugB_name', 'drugB_conc', 'cell_line', 'single_resp_1', 'single_resp_2', 'target']
Drug numeric feature shape: (95, 6890)
Cell numeric feature shape: (35, 1000)
[Warning] drug